In [ ]:
!pip install jax flax distrax chex wandb pillow numpy optax

In [ ]:
!mkdir -p envs

In [ ]:
%%writefile envs/navix_wrapper.py
import jax
import jax.numpy as jnp
import numpy as np
from navix import observations

UNSET_POSITION = jnp.array([-1, -1], dtype=jnp.int32)


class NavixGridWrapper:
    """Expose Navix single-agent environments with the Seer/Doer observation split."""

    SEER_NAV_PHASE = 0
    COMMUNICATION_PHASE = 1

    def __init__(
        self,
        env,
        progress_reward_scale: float = 0.1,
        min_start_distance: float = 0.0,
        step_penalty: float = 0.0,
        bump_penalty: float = 0.1,
        doer_perception_level: int = 0,
    ):
        self._env = env
        self.progress_reward_scale = progress_reward_scale
        self.min_start_distance = jnp.asarray(min_start_distance, dtype=jnp.float32)
        self.step_penalty = jnp.asarray(step_penalty, dtype=jnp.float32)
        self.bump_penalty = jnp.asarray(bump_penalty, dtype=jnp.float32)
        self.doer_perception_level = int(doer_perception_level)

    @property
    def num_actions(self) -> int:
        # Navigation-only action space: turn left, turn right, move forward.
        return 3

    @staticmethod
    def player_position(timestep) -> jnp.ndarray:
        return timestep.state.get_player().position.astype(jnp.int32)

    @staticmethod
    def goal_position(timestep) -> jnp.ndarray:
        return timestep.state.get_goals().position[0].astype(jnp.int32)

    @staticmethod
    def _position_matches(actual_position: jnp.ndarray, target_position: jnp.ndarray) -> jnp.ndarray:
        return jnp.logical_or(
            jnp.any(target_position < 0),
            jnp.all(actual_position == target_position),
        )

    def _apply_curriculum_positions(
        self,
        timestep,
        fixed_goal_position: jnp.ndarray,
        fixed_start_position: jnp.ndarray,
    ):
        def set_goal(state):
            goals = state.get_goals()
            updated_goal_positions = goals.position.at[0].set(
                fixed_goal_position.astype(goals.position.dtype)
            )
            return state.set_goals(goals.replace(position=updated_goal_positions))

        def set_start(state):
            player = state.get_player()
            return state.set_player(
                player.replace(
                    position=fixed_start_position.astype(player.position.dtype)
                )
            )

        state = jax.lax.cond(
            jnp.all(fixed_goal_position >= 0),
            set_goal,
            lambda state: state,
            timestep.state,
        )
        state = jax.lax.cond(
            jnp.all(fixed_start_position >= 0),
            set_start,
            lambda state: state,
            state,
        )

        return timestep.replace(state=state)

    def _split_observations(
        self,
        timestep,
        vision_radius: jnp.ndarray,
        control_mode: jnp.ndarray,
    ):
        state = timestep.state
        global_map = observations.symbolic(state).astype(jnp.float32)
        full_local_view = observations.symbolic_first_person(state).astype(jnp.float32)

        player = state.get_player()
        goal = state.get_goals()
        center_row = full_local_view.shape[0] // 2
        center_col = full_local_view.shape[1] // 2
        local_view_3x3 = jax.lax.dynamic_slice(
            full_local_view,
            (center_row - 1, center_col - 1, 0),
            (3, 3, full_local_view.shape[-1]),
        )

        symbolic_state = jnp.array(
            [
                player.position[0],
                player.position[1],
                player.direction,
                player.pocket,
                goal.position[0, 0],
                goal.position[0, 1],
                (control_mode == self.SEER_NAV_PHASE).astype(jnp.float32),
            ],
            dtype=jnp.float32,
        )

        proprioception_full = jnp.array(
            [
                player.position[0],
                player.position[1],
                player.direction,
                player.pocket,
            ],
            dtype=jnp.float32,
        )

        if self.doer_perception_level == 0:
            local_view = local_view_3x3
            proprioception = proprioception_full
        elif self.doer_perception_level == 1:
            local_view = jnp.zeros_like(local_view_3x3)
            local_view = local_view.at[0, 1].set(local_view_3x3[0, 1])
            proprioception = proprioception_full
        elif self.doer_perception_level == 2:
            local_view = jnp.zeros_like(local_view_3x3)
            proprioception = jnp.array(
                [
                    player.position[0],
                    player.position[1],
                    0.0,
                    0.0,
                ],
                dtype=jnp.float32,
            )
        elif self.doer_perception_level == 3:
            local_view = jnp.zeros_like(local_view_3x3)
            proprioception = jnp.zeros_like(proprioception_full)
        else:
            raise ValueError(
                f"Unsupported doer_perception_level={self.doer_perception_level}. "
                "Use 0, 1, 2, or 3."
            )

        return {
            "global_map": global_map,
            "symbolic_state": symbolic_state,
            "local_view": local_view,
            "proprioception": proprioception,
        }

    @staticmethod
    def _goal_distance(state) -> jnp.ndarray:
        player = state.get_player()
        goal = state.get_goals()
        return jnp.abs(player.position - goal.position[0]).sum().astype(jnp.float32)

    def reset(
        self,
        key: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        timestep = self._env.reset(key)
        timestep = self._apply_curriculum_positions(
            timestep,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )

        distance = self._goal_distance(timestep.state)

        def cond_fn(carry):
            _, _, goal_distance = carry
            return goal_distance < self.min_start_distance

        def body_fn(carry):
            rng, _, _ = carry
            rng, sample_key = jax.random.split(rng)
            sampled_timestep = self._env.reset(sample_key)
            sampled_timestep = self._apply_curriculum_positions(
                sampled_timestep,
                fixed_goal_position=fixed_goal_position,
                fixed_start_position=fixed_start_position,
            )
            goal_distance = self._goal_distance(sampled_timestep.state)
            return rng, sampled_timestep, goal_distance

        _, timestep, _ = jax.lax.while_loop(
            cond_fn,
            body_fn,
            (key, timestep, distance),
        )
        obs = self._split_observations(timestep, vision_radius, control_mode)
        return obs, timestep

    def reset_batch(
        self,
        keys: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        return jax.vmap(self.reset, in_axes=(0, None, None, None, None))(
            keys,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        )

    def step(
        self,
        key: jnp.ndarray,
        timestep,
        action: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        def reset_branch(_):
            reset_obs, reset_timestep = self.reset(
                key,
                vision_radius=vision_radius,
                control_mode=control_mode,
                fixed_goal_position=fixed_goal_position,
                fixed_start_position=fixed_start_position,
            )
            reward = jnp.asarray(0.0, dtype=jnp.float32)
            done = jnp.asarray(False)
            info = {
                "return": reset_timestep.info.get("return", reward),
                "task_reward": reward,
                "progress_reward": reward,
                "step_penalty": reward,
                "bump_penalty": reward,
                "goal_distance": self._goal_distance(reset_timestep.state),
            }
            return reset_obs, reset_timestep, reward, done, info

        def step_branch(_):
            old_distance = self._goal_distance(timestep.state)
            old_position = timestep.state.get_player().position
            next_timestep = self._env.step(timestep, action)
            new_distance = self._goal_distance(next_timestep.state)
            new_position = next_timestep.state.get_player().position
            obs = self._split_observations(next_timestep, vision_radius, control_mode)
            task_reward = next_timestep.reward.astype(jnp.float32)
            progress_reward = (old_distance - new_distance) * self.progress_reward_scale
            step_penalty = self.step_penalty
            bumped = jnp.logical_and(
                action == 2,
                jnp.all(new_position == old_position),
            )
            bump_penalty = jnp.where(
                bumped,
                self.bump_penalty,
                jnp.asarray(0.0, dtype=jnp.float32),
            )
            reward = task_reward + progress_reward - step_penalty - bump_penalty
            done = next_timestep.is_done()
            info = dict(next_timestep.info)
            info["task_reward"] = task_reward
            info["progress_reward"] = progress_reward
            info["step_penalty"] = step_penalty
            info["bump_penalty"] = bump_penalty
            info["goal_distance"] = new_distance
            return obs, next_timestep, reward, done, info

        return jax.lax.cond(timestep.is_done(), reset_branch, step_branch, operand=None)

    def step_batch(
        self,
        keys: jnp.ndarray,
        timesteps,
        actions: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        return jax.vmap(self.step, in_axes=(0, 0, 0, None, None, None, None))(
            keys,
            timesteps,
            actions,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        )

    def render(
        self,
        timestep,
        control_mode: int = COMMUNICATION_PHASE,
    ):
        frame = np.asarray(observations.rgb(timestep.state)).copy()
        grid = np.asarray(observations.symbolic(timestep.state))
        player = np.asarray(timestep.state.get_player().position).astype(np.int32)
        tile_h = max(frame.shape[0] // grid.shape[0], 1)
        tile_w = max(frame.shape[1] // grid.shape[1], 1)
        row, col = int(player[0]), int(player[1])
        y0 = row * tile_h + tile_h // 4
        y1 = min((row + 1) * tile_h - tile_h // 4, frame.shape[0])
        x0 = col * tile_w + tile_w // 4
        x1 = min((col + 1) * tile_w - tile_w // 4, frame.shape[1])
        color = (
            np.array([32, 96, 224], dtype=np.uint8)
            if control_mode == self.SEER_NAV_PHASE
            else np.array([0, 0, 0], dtype=np.uint8)
        )
        frame[y0:y1, x0:x1] = color
        return frame


In [ ]:
!mkdir -p envs

In [ ]:
%%writefile envs/two_doer_grid.py
import jax
import jax.numpy as jnp
import numpy as np
from flax import struct

UNSET_TWO_DOER_POSITIONS = jnp.full((2, 2), -1, dtype=jnp.int32)


@struct.dataclass
class TwoDoerState:
    positions: jnp.ndarray
    goals: jnp.ndarray
    step_count: jnp.ndarray
    done: jnp.ndarray
    target_items: jnp.ndarray
    shuffled_menus: jnp.ndarray
    selected_correctly: jnp.ndarray
    has_selected: jnp.ndarray
    has_arrived: jnp.ndarray


class TwoDoerBottleneckEnv:
    """Two embodied Doers must swap sides through a single choke point."""

    def __init__(
        self,
        grid_height: int = 10,
        grid_width: int = 12,
        local_view_size: int = 3,
        corridor_length: int = 3,
        max_steps: int = 48,
        progress_reward_scale: float = 0.05,
        goal_reward: float = 1.0,
        arrival_reward: float = 0.5,
        step_penalty: float = 0.03,
        wall_penalty: float = 0.02,
        collision_penalty: float = 0.05,
        doer_perception_level: int = 2,
        render_tile_size: int = 24,
    ):
        if grid_height < 8:
            raise ValueError("grid_height must be >= 8.")
        if grid_width < 10:
            raise ValueError("grid_width must be >= 10.")
        if local_view_size % 2 == 0:
            raise ValueError("local_view_size must be odd.")
        if corridor_length < 1:
            raise ValueError("corridor_length must be >= 1.")
        if grid_width - corridor_length - 2 < 4:
            raise ValueError("grid_width is too small for the requested corridor_length.")

        self.grid_height = int(grid_height)
        self.grid_width = int(grid_width)
        self.local_view_size = int(local_view_size)
        self.corridor_length = int(corridor_length)
        self.max_steps = int(max_steps)
        self.progress_reward_scale = jnp.asarray(progress_reward_scale, dtype=jnp.float32)
        self.goal_reward = jnp.asarray(goal_reward, dtype=jnp.float32)
        self.arrival_reward = jnp.asarray(arrival_reward, dtype=jnp.float32)
        self.step_penalty = jnp.asarray(step_penalty, dtype=jnp.float32)
        self.wall_penalty = jnp.asarray(wall_penalty, dtype=jnp.float32)
        self.collision_penalty = jnp.asarray(collision_penalty, dtype=jnp.float32)
        self.doer_perception_level = int(doer_perception_level)
        self.render_tile_size = int(render_tile_size)
        self.num_doers = 2
        self._view_radius = self.local_view_size // 2
        self._inner_width = self.grid_width - 2
        self._room_width = (self._inner_width - self.corridor_length) // 2
        self._extra_width = self._inner_width - self.corridor_length - 2 * self._room_width
        self._left_room_start_col = 1
        self._left_room_end_col = self._left_room_start_col + self._room_width
        self._corridor_row = self.grid_height // 2
        self._corridor_start_col = self._left_room_end_col
        self._corridor_end_col = self._corridor_start_col + self.corridor_length
        self._right_room_start_col = self._corridor_end_col
        self._right_room_end_col = self._right_room_start_col + self._room_width
        self._extra_wall_start_col = self._right_room_end_col
        self._extra_wall_end_col = self.grid_width - 1
        self._left_col = 1
        self._right_col = self._right_room_end_col - 1
        self._candidate_rows = jnp.asarray([1, 2, self.grid_height - 3, self.grid_height - 2], dtype=jnp.int32)
        self._wall_map = self._build_wall_map()
        self._goal_colors = jnp.asarray(
            [
                [0.90, 0.25, 0.25],
                [0.20, 0.70, 0.30],
            ],
            dtype=jnp.float32,
        )
        self._agent_colors = jnp.asarray(
            [
                [0.85, 0.10, 0.10],
                [0.05, 0.45, 0.95],
            ],
            dtype=jnp.float32,
        )
        self.item_bank = self._build_item_bank()

    @property
    def num_actions(self) -> int:
        # stay, up, right, down, left, pick_0, pick_1, pick_2, pick_3
        return 9
        
    def _build_item_bank(self) -> jnp.ndarray:
        colors = jnp.asarray([
            [1.0, 0.2, 0.2],
            [0.2, 1.0, 0.2],
            [0.2, 0.4, 1.0],
            [1.0, 1.0, 0.2],
        ], dtype=jnp.float32)
        shape0 = jnp.ones((5, 5), dtype=jnp.float32)
        shape1 = jnp.zeros((5, 5), dtype=jnp.float32)
        shape1 = shape1.at[2, 1:4].set(1.0)
        shape1 = shape1.at[1:4, 2].set(1.0)
        shape2 = jnp.zeros((5, 5), dtype=jnp.float32)
        shape2 = shape2.at[1, 1].set(1.0)
        shape2 = shape2.at[2, 2].set(1.0)
        shape2 = shape2.at[3, 3].set(1.0)
        shape2 = shape2.at[1, 3].set(1.0)
        shape2 = shape2.at[3, 1].set(1.0)
        shape3 = jnp.ones((5, 5), dtype=jnp.float32)
        shape3 = shape3.at[1:4, 1:4].set(0.0)
        shapes = jnp.stack([shape0, shape1, shape2, shape3])
        bank = jnp.zeros((16, 5, 5, 3), dtype=jnp.float32)
        for c in range(4):
            for s in range(4):
                bank = bank.at[c * 4 + s].set(shapes[s, :, :, None] * colors[c])
        return bank

    def _build_wall_map(self) -> jnp.ndarray:
        wall_map = jnp.zeros((self.grid_height, self.grid_width), dtype=bool)
        wall_map = wall_map.at[0, :].set(True)
        wall_map = wall_map.at[-1, :].set(True)
        wall_map = wall_map.at[:, 0].set(True)
        wall_map = wall_map.at[:, -1].set(True)
        corridor_cols = jnp.arange(self._corridor_start_col, self._corridor_end_col)
        wall_map = wall_map.at[:, corridor_cols].set(True)
        wall_map = wall_map.at[self._corridor_row, corridor_cols].set(False)
        if self._extra_width > 0:
            extra_cols = jnp.arange(self._extra_wall_start_col, self._extra_wall_end_col)
            wall_map = wall_map.at[:, extra_cols].set(True)
        return wall_map

    @staticmethod
    def _manhattan_distance(positions: jnp.ndarray, goals: jnp.ndarray) -> jnp.ndarray:
        return jnp.abs(positions - goals).sum(axis=-1).astype(jnp.float32)

    def _compose_global_map(self, state: TwoDoerState) -> jnp.ndarray:
        global_map = jnp.zeros((self.grid_height, self.grid_width, 5), dtype=jnp.float32)
        global_map = global_map.at[:, :, 0].set(self._wall_map.astype(jnp.float32))
        global_map = global_map.at[state.positions[0, 0], state.positions[0, 1], 1].set(1.0)
        global_map = global_map.at[state.positions[1, 0], state.positions[1, 1], 2].set(1.0)
        global_map = global_map.at[state.goals[0, 0], state.goals[0, 1], 3].set(1.0)
        global_map = global_map.at[state.goals[1, 0], state.goals[1, 1], 4].set(1.0)
        return global_map

    def _extract_local_views(self, global_map: jnp.ndarray, positions: jnp.ndarray) -> jnp.ndarray:
        padded_map = jnp.pad(
            global_map,
            (
                (self._view_radius, self._view_radius),
                (self._view_radius, self._view_radius),
                (0, 0),
            ),
        )

        def slice_agent_view(position):
            row = position[0]
            col = position[1]
            return jax.lax.dynamic_slice(
                padded_map,
                (row, col, 0),
                (self.local_view_size, self.local_view_size, global_map.shape[-1]),
            )

        return jax.vmap(slice_agent_view)(positions)

    def _split_observations(self, state: TwoDoerState):
        global_map = self._compose_global_map(state)
        full_local_views = self._extract_local_views(global_map, state.positions)
        goals_reached = jnp.all(state.positions == state.goals, axis=-1).astype(jnp.float32)
        agent_identity = jnp.eye(self.num_doers, dtype=jnp.float32)
        position_features = jnp.stack(
            [
                state.positions[:, 0].astype(jnp.float32) / float(self.grid_height - 1),
                state.positions[:, 1].astype(jnp.float32) / float(self.grid_width - 1),
            ],
            axis=-1,
        )
        proprioception_full = jnp.concatenate(
            [position_features, agent_identity, goals_reached[:, None]],
            axis=-1,
        )
        if self.doer_perception_level == 2:
            local_views = jnp.zeros_like(full_local_views)
            proprioceptions = proprioception_full
        elif self.doer_perception_level == 3:
            local_views = jnp.zeros_like(full_local_views)
            proprioceptions = jnp.concatenate(
                [
                    jnp.zeros_like(position_features),
                    agent_identity,
                    jnp.zeros_like(goals_reached[:, None]),
                ],
                axis=-1,
            )
        else:
            raise ValueError(
                f"Unsupported doer_perception_level={self.doer_perception_level}. "
                "Use 2 or 3."
            )
        symbolic_state = jnp.concatenate(
            [
                jnp.asarray(
                    [
                        state.positions[0, 0] / float(self.grid_height - 1),
                        state.positions[0, 1] / float(self.grid_width - 1),
                        state.positions[1, 0] / float(self.grid_height - 1),
                        state.positions[1, 1] / float(self.grid_width - 1),
                    ],
                    dtype=jnp.float32,
                ),
                jnp.asarray(
                    [
                        state.goals[0, 0] / float(self.grid_height - 1),
                        state.goals[0, 1] / float(self.grid_width - 1),
                        state.goals[1, 0] / float(self.grid_height - 1),
                        state.goals[1, 1] / float(self.grid_width - 1),
                    ],
                    dtype=jnp.float32,
                ),
                goals_reached,
                jnp.asarray(
                    [state.step_count.astype(jnp.float32) / float(self.max_steps)],
                    dtype=jnp.float32,
                ),
            ],
            axis=0,
        )
        
        target_images = self.item_bank[state.target_items]
        menu_images = self.item_bank[state.shuffled_menus]
        menu_images = jnp.where(goals_reached[:, None, None, None, None], menu_images, 0.0)

        return {
            "global_map": global_map,
            "symbolic_state": symbolic_state,
            "local_views": local_views,
            "proprioceptions": proprioceptions,
            "target_images": target_images,
            "menu_images": menu_images,
        }

    def _goals_from_positions(self, positions: jnp.ndarray) -> jnp.ndarray:
        return jnp.asarray(
            [
                [positions[0, 0], self._right_col],
                [positions[1, 0], self._left_col],
            ],
            dtype=jnp.int32,
        )

    def reset(
        self,
        key: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        row_key_a, row_key_b = jax.random.split(key)
        row_a = self._candidate_rows[jax.random.randint(row_key_a, (), 0, self._candidate_rows.shape[0])]
        row_b = self._candidate_rows[jax.random.randint(row_key_b, (), 0, self._candidate_rows.shape[0])]
        sampled_positions = jnp.asarray(
            [
                [row_a, self._left_col],
                [row_b, self._right_col],
            ],
            dtype=jnp.int32,
        )
        positions = jnp.where(
            jnp.all(fixed_positions >= 0),
            fixed_positions.astype(jnp.int32),
            sampled_positions,
        )
        goals = self._goals_from_positions(positions)
        
        key_target_a, key_target_b, key_menu_a, key_menu_b = jax.random.split(key, 4)
        target_a = jax.random.randint(key_target_a, (), 0, 16)
        target_b = jax.random.randint(key_target_b, (), 0, 16)
        
        def make_menu(rng_key, target):
            p = jnp.where(jnp.arange(16) == target, 0.0, 1.0 / 15.0)
            distractors = jax.random.choice(rng_key, jnp.arange(16), shape=(3,), replace=False, p=p)
            menu = jnp.concatenate([jnp.array([target]), distractors])
            _, shuffle_key = jax.random.split(rng_key)
            return jax.random.permutation(shuffle_key, menu)
            
        menu_a = make_menu(key_menu_a, target_a)
        menu_b = make_menu(key_menu_b, target_b)
        
        state = TwoDoerState(
            positions=positions,
            goals=goals,
            step_count=jnp.asarray(0, dtype=jnp.int32),
            done=jnp.asarray(False),
            target_items=jnp.array([target_a, target_b]),
            shuffled_menus=jnp.stack([menu_a, menu_b]),
            selected_correctly=jnp.array([False, False]),
            has_selected=jnp.array([False, False]),
            has_arrived=jnp.array([False, False]),
        )
        return self._split_observations(state), state

    def reset_batch(
        self,
        keys: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        return jax.vmap(self.reset, in_axes=(0, None))(keys, fixed_positions)

    def _resolve_actions(self, positions: jnp.ndarray, actions: jnp.ndarray):
        deltas = jnp.asarray(
            [
                [0, 0],
                [-1, 0],
                [0, 1],
                [1, 0],
                [0, -1],
            ],
            dtype=jnp.int32,
        )
        raw_targets = positions + deltas[actions]
        target_walls = self._wall_map[raw_targets[:, 0], raw_targets[:, 1]]
        wall_hits = jnp.logical_and(actions != 0, target_walls)
        proposed = jnp.where(target_walls[:, None], positions, raw_targets)

        same_target = jnp.all(proposed[0] == proposed[1])
        swap_positions = jnp.logical_and(
            jnp.all(proposed[0] == positions[1]),
            jnp.all(proposed[1] == positions[0]),
        )
        a_into_b = jnp.logical_and(
            jnp.all(proposed[0] == positions[1]),
            jnp.all(proposed[1] == positions[1]),
        )
        b_into_a = jnp.logical_and(
            jnp.all(proposed[1] == positions[0]),
            jnp.all(proposed[0] == positions[0]),
        )
        collision = jnp.logical_or(jnp.logical_or(same_target, swap_positions), jnp.logical_or(a_into_b, b_into_a))

        collision_blocks = jnp.asarray(
            [
                jnp.logical_or(same_target, jnp.logical_or(swap_positions, a_into_b)),
                jnp.logical_or(same_target, jnp.logical_or(swap_positions, b_into_a)),
            ],
            dtype=bool,
        )
        final_positions = jnp.where(collision_blocks[:, None], positions, proposed)
        return final_positions, wall_hits, collision_blocks

    def step(
        self,
        key: jnp.ndarray,
        state: TwoDoerState,
        actions: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        def reset_branch(_):
            reset_obs, reset_state = self.reset(key, fixed_positions=fixed_positions)
            zeros = jnp.asarray(0.0, dtype=jnp.float32)
            info = {
                "task_reward": zeros,
                "progress_reward_per_doer": jnp.zeros((self.num_doers,), dtype=jnp.float32),
                "step_penalty": zeros,
                "wall_penalty": zeros,
                "collision_penalty": zeros,
                "goal_distance": self._manhattan_distance(reset_state.positions, reset_state.goals),
                "success": jnp.asarray(False),
                "failed": jnp.asarray(False),
            }
            return reset_obs, reset_state, zeros, jnp.asarray(False), info

        def step_branch(_):
            nav_actions = jnp.where(actions < 5, actions, 0)
            select_actions = actions - 5
            is_selecting = actions >= 5
            at_goal = jnp.all(state.positions == state.goals, axis=-1)

            valid_selection = jnp.logical_and(is_selecting, at_goal)
            valid_selection = jnp.logical_and(valid_selection, ~state.has_selected)

            chosen_item = jnp.take_along_axis(state.shuffled_menus, select_actions[:, None], axis=1)[:, 0]
            is_correct = chosen_item == state.target_items
            
            new_has_selected = jnp.logical_or(state.has_selected, valid_selection)
            new_selected_correctly = jnp.logical_or(state.selected_correctly, jnp.logical_and(valid_selection, is_correct))
            
            old_distances = self._manhattan_distance(state.positions, state.goals)
            next_positions, wall_hits, collision_blocks = self._resolve_actions(state.positions, nav_actions)
            next_positions = jnp.where(is_selecting[:, None], state.positions, next_positions)
            
            new_distances = self._manhattan_distance(next_positions, state.goals)
            progress_reward_per_doer = (old_distances - new_distances) * self.progress_reward_scale
            progress_reward = progress_reward_per_doer.sum()
            wall_penalty = self.wall_penalty * wall_hits.astype(jnp.float32).sum()
            collision_penalty = (
                self.collision_penalty * collision_blocks.astype(jnp.float32).sum()
            )
            
            arrived_this_step = jnp.logical_and(
                jnp.all(next_positions == state.goals, axis=-1),
                ~state.has_arrived
            )
            arrival_reward = (arrived_this_step.astype(jnp.float32) * self.arrival_reward).sum()
            new_has_arrived = jnp.logical_or(state.has_arrived, arrived_this_step)
            
            success = jnp.all(new_selected_correctly)
            failed = jnp.any(jnp.logical_and(new_has_selected, ~new_selected_correctly))
            
            task_reward = jnp.where(success, self.goal_reward, jnp.asarray(0.0, dtype=jnp.float32))
            reward = (
                task_reward
                + progress_reward
                + arrival_reward
                - self.step_penalty
                - wall_penalty
                - collision_penalty
            )
            next_state = TwoDoerState(
                positions=next_positions,
                goals=state.goals,
                step_count=state.step_count + 1,
                done=jnp.logical_or(jnp.logical_or(success, failed), state.step_count + 1 >= self.max_steps),
                target_items=state.target_items,
                shuffled_menus=state.shuffled_menus,
                selected_correctly=new_selected_correctly,
                has_selected=new_has_selected,
                has_arrived=new_has_arrived,
            )
            info = {
                "task_reward": task_reward,
                "progress_reward_per_doer": progress_reward_per_doer,
                "step_penalty": self.step_penalty,
                "wall_penalty": wall_penalty,
                "collision_penalty": collision_penalty,
                "goal_distance": new_distances,
                "success": success,
                "failed": failed,
            }
            return self._split_observations(next_state), next_state, reward, next_state.done, info

        return jax.lax.cond(state.done, reset_branch, step_branch, operand=None)

    def step_batch(
        self,
        keys: jnp.ndarray,
        states: TwoDoerState,
        actions: jnp.ndarray,
        fixed_positions: jnp.ndarray = UNSET_TWO_DOER_POSITIONS,
    ):
        return jax.vmap(self.step, in_axes=(0, 0, 0, None))(keys, states, actions, fixed_positions)

    def render(self, state: TwoDoerState) -> np.ndarray:
        tile = self.render_tile_size
        frame = np.ones((self.grid_height * tile, self.grid_width * tile, 3), dtype=np.float32) * 0.96

        wall_color = np.array([0.18, 0.18, 0.22], dtype=np.float32)
        corridor_color = np.array([0.98, 0.88, 0.45], dtype=np.float32)

        wall_map = np.asarray(self._wall_map)
        for row in range(self.grid_height):
            for col in range(self.grid_width):
                y0 = row * tile
                y1 = (row + 1) * tile
                x0 = col * tile
                x1 = (col + 1) * tile
                if wall_map[row, col]:
                    frame[y0:y1, x0:x1] = wall_color

        for col in range(self._corridor_start_col, self._corridor_end_col):
            y0 = self._corridor_row * tile
            y1 = (self._corridor_row + 1) * tile
            x0 = col * tile
            x1 = (col + 1) * tile
            frame[y0:y1, x0:x1] = corridor_color

        for agent_idx in range(self.num_doers):
            goal_row, goal_col = np.asarray(state.goals[agent_idx]).tolist()
            y0 = goal_row * tile
            x0 = goal_col * tile
            inset = max(tile // 4, 2)
            frame[
                y0 + inset:(goal_row + 1) * tile - inset,
                x0 + inset:(goal_col + 1) * tile - inset,
            ] = np.asarray(self._agent_colors[agent_idx])

        yy, xx = np.mgrid[0:tile, 0:tile]
        left_triangle = np.logical_and(xx >= tile // 5, np.abs(yy - tile // 2) <= xx - tile // 5)
        right_triangle = np.logical_and(
            xx <= tile - tile // 5 - 1,
            np.abs(yy - tile // 2) <= (tile - tile // 5 - 1) - xx,
        )

        for agent_idx in range(self.num_doers):
            row, col = np.asarray(state.positions[agent_idx]).tolist()
            y0 = row * tile
            x0 = col * tile
            tile_view = frame[y0:(row + 1) * tile, x0:(col + 1) * tile]
            triangle_mask = right_triangle if agent_idx == 0 else left_triangle
            tile_view[triangle_mask] = np.asarray(self._agent_colors[agent_idx])
            frame[y0:(row + 1) * tile, x0:(col + 1) * tile] = tile_view

        # Render Item Targets at goals
        bank = np.asarray(self.item_bank)
        for agent_idx in range(self.num_doers):
            target_item = int(np.asarray(state.target_items)[agent_idx])
            target_img = bank[target_item] # (5, 5, 3)
            # scale 5x5 to inset region
            goal_row, goal_col = np.asarray(state.goals[agent_idx]).tolist()
            y0 = goal_row * tile
            x0 = goal_col * tile
            # upscale target_img to 15x15
            target_upscaled = np.repeat(np.repeat(target_img, 3, axis=0), 3, axis=1)
            offset_y, offset_x = (tile - 15) // 2, (tile - 15) // 2
            
            # Place target in center of goal
            frame[
                y0 + offset_y : y0 + offset_y + 15, 
                x0 + offset_x : x0 + offset_x + 15
            ] = target_upscaled
            
            # Render menu options when goal is reached
            if bool(np.asarray(state.positions[agent_idx] == state.goals[agent_idx]).all()):
                menus = np.asarray(state.shuffled_menus[agent_idx])
                for m_idx, option_id in enumerate(menus):
                    opt_img = bank[option_id]
                    opt_upscaled = np.repeat(np.repeat(opt_img, 3, axis=0), 3, axis=1)
                    # draw menu below the grid or adjacent
                    my0 = y0 - (m_idx + 1) * tile if agent_idx == 1 else y0 + (m_idx + 1) * tile
                    # clip boundary
                    my0 = max(0, min(my0, (self.grid_height - 1) * tile))
                    frame[my0 + offset_y : my0 + offset_y + 15, x0 + offset_x : x0 + offset_x + 15] = opt_upscaled

        return (frame * 255.0).astype(np.uint8)


In [ ]:
!mkdir -p envs

In [ ]:
%%writefile envs/wrappers.py
import jax
import jax.numpy as jnp
import numpy as np
from typing import Tuple, Dict, Any
from jaxmarl.viz.overcooked_visualizer import OvercookedVisualizer

class AsymmetricOvercookedWrapper:
    """
    Wraps a jaxmarl Overcooked environment to enforce the Seer-Doer split.
    Guarantees strict sensory deprivation for the Doer agent.
    """
    def __init__(self, env):
        self._env = env
        self._visualizer = OvercookedVisualizer()
        # In standard jaxmarl Overcooked, agents are usually 'agent_0' and 'agent_1'
        self.seer_id = 'agent_0'
        self.doer_id = 'agent_1'

    @property
    def num_actions(self) -> int:
        return self._env.action_space(self.doer_id).n

    def _split_observations(self, state: Any, raw_obs: Dict[str, jnp.ndarray], vision_radius: jnp.ndarray) -> Dict[str, Any]:
        """
        Transforms symmetric jaxmarl observations into structurally asymmetric inputs.
        Supports vision curriculum via vision_radius.
        """
        # 1. Seer (Prefrontal Cortex) Data Extraction
        # The Seer requires the Global Map View and Symbolic States[cite: 131, 132].
        # Depending on the specific jaxmarl layout, the raw observation might already 
        # be the global grid, or we might need to extract it from the environment state.
        global_map = state.maze_map # Example extraction of the underlying grid
        
        # Placeholder for symbolic state (e.g., recipe requirements, time remaining)
        # Using time and current agent inventory as symbolic state
        symbolic_state = jnp.array([state.time, state.agent_inv[0], state.agent_inv[1]]) 

        # 2. Doer (Motor Cortex) Data Extraction
        # The Doer requires an Egocentric 3x3 grid (or total blindness) and proprioception[cite: 137, 138].
        doer_idx = 1 if self.doer_id == 'agent_1' else 0
        doer_pos = state.agent_pos[doer_idx]
        doer_dir = state.agent_dir[doer_idx]
        
        local_view = raw_obs[self.doer_id]
        
        # Local view masking based on vision_radius
        # raw_obs is map shape (H, W, C).
        H, W = local_view.shape[0], local_view.shape[1]
        yy, xx = jnp.meshgrid(jnp.arange(H), jnp.arange(W), indexing='ij')
        
        # In jaxmarl, agent_pos is [x, y]. yy is height index (y), xx is width index (x).
        doer_x, doer_y = state.agent_pos[doer_idx][0], state.agent_pos[doer_idx][1]
        dist = jnp.maximum(jnp.abs(xx - doer_x), jnp.abs(yy - doer_y)) # Chebyshev distance
        mask = (dist <= vision_radius)[..., None]
        local_view = jnp.where(mask, local_view, 0)

        # Proprioception: e.g., Is the Doer holding an onion or a plate? [cite: 138]
        proprioception = jnp.array([state.agent_inv[doer_idx]])

        # 3. Restructure for our custom rollout loop
        return {
            "global_map": global_map,
            "symbolic_state": symbolic_state,
            "local_view": local_view,
            "proprioception": proprioception
        }

    def reset(self, key: jnp.ndarray, vision_radius: jnp.ndarray = jnp.array(2.0)) -> Tuple[Dict[str, Any], Any]:
        """Resets the environment and returns the asymmetric observation dictionary."""
        raw_obs, state = self._env.reset(key)
        asymmetric_obs = self._split_observations(state, raw_obs, vision_radius)
        return asymmetric_obs, state

    def reset_batch(
        self,
        keys: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(2.0)
    ) -> Tuple[Dict[str, Any], Any]:
        """Vectorized reset across a batch of environment keys."""
        return jax.vmap(self.reset, in_axes=(0, None))(keys, vision_radius)

    def step(
        self, 
        key: jnp.ndarray, 
        state: Any, 
        doer_action: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(2.0)
    ) -> Tuple[Dict[str, Any], Any, jnp.ndarray, jnp.ndarray, Dict]:
        """
        Steps the environment forward. 
        Notice that only the Doer submits a physical action.
        """
        # The Seer has zero physical agency[cite: 80]. 
        # Its "action" is the message passed directly to the Doer's neural network 
        # during the rollout loop, not to the environment simulator.
        env_actions = {
            self.seer_id: jnp.array(4), # 4 is STAY action in jaxmarl overcooked
            self.doer_id: doer_action
        }
        
        raw_obs, next_state, rewards, dones, info = self._env.step(key, state, env_actions)
        
        asymmetric_obs = self._split_observations(next_state, raw_obs, vision_radius)
        
        # Immediate Reward Shaping: Sub-Goal for picking up or dropping an item
        # Helps the Doer associate the Seer's message with object interaction
        doer_idx = 1 if self.doer_id == 'agent_1' else 0
        old_inv = state.agent_inv[doer_idx]
        new_inv = next_state.agent_inv[doer_idx]
        
        # Reward +0.1 for picking up an object
        picked_up = jnp.logical_and(old_inv == 0, new_inv != 0)
        pickup_reward = jnp.where(picked_up, 0.1, 0.0)
        
        # Reward +0.1 for dropping an object (e.g. in pot or counter)
        dropped = jnp.logical_and(old_inv != 0, new_inv == 0)
        drop_reward = jnp.where(dropped, 0.1, 0.0)
        
        # In a fully cooperative game like Overcooked, rewards are usually shared
        shared_reward = rewards[self.doer_id] + pickup_reward + drop_reward
        shared_done = dones['__all__']
        
        return asymmetric_obs, next_state, shared_reward, shared_done, info

    def step_batch(
        self,
        keys: jnp.ndarray,
        states: Any,
        doer_actions: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(2.0)
    ) -> Tuple[Dict[str, Any], Any, jnp.ndarray, jnp.ndarray, Dict]:
        """Vectorized step across a batch of environment states and actions."""
        return jax.vmap(self.step, in_axes=(0, 0, 0, None))(keys, states, doer_actions, vision_radius)

    def render(self, state: Any) -> np.ndarray:
        """Render a single environment state to an RGB frame."""
        padding = self._env.agent_view_size - 2
        grid = np.asarray(state.maze_map[padding:-padding, padding:-padding, :])
        return OvercookedVisualizer._render_grid(
            grid,
            tile_size=32,
            highlight_mask=None,
            agent_dir_idx=np.asarray(state.agent_dir_idx),
            agent_inv=np.asarray(state.agent_inv),
        )


In [ ]:
!mkdir -p models

In [ ]:
%%writefile models/fsq.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence

class FSQ(nn.Module):
    """
    Finite Scalar Quantization (FSQ) module.
    Projects a continuous vector into a discrete hypercube.
    """
    # A sequence of integers defining the number of levels (L) per dimension (d).
    # e.g., levels=[5, 5, 5] means d=3 dimensions, each with 5 discrete levels.
    levels: Sequence[int]

    @nn.compact
    def __call__(self, z: jnp.ndarray) -> jnp.ndarray:
        """
        Args:
            z: The continuous "thought vector" from the Seer's reasoning module.
               Expected shape: (batch_size, ..., d) where d == len(self.levels)
        Returns:
            z_ste: The quantized vector with gradients preserved via STE.
        """
        levels = jnp.asarray(self.levels, dtype=z.dtype)
        
        # 1. Bounding: Restrict the unbounded input z to the range [-1, 1].
        # We use tanh as a standard, proven method for this projection.
        z_bound = jnp.tanh(z)
        
        # 2. Scaling: Map the [-1, 1] range to the grid [0, L - 1].
        # E.g., for L=5, this maps [-1, 1] to [0, 4].
        half_width = (levels - 1) / 2.0
        z_scaled = z_bound * half_width + half_width
        
        # 3. Quantization: Snap to the nearest integer grid point.
        if self.has_rng('noise'):
            # Inject uniform noise during training to "shake" the FSQ and prevent early mode collapse
            noise = jax.random.uniform(self.make_rng('noise'), z_scaled.shape, minval=-0.2, maxval=0.2)
            z_quantized = jnp.round(z_scaled + noise)
        else:
            z_quantized = jnp.round(z_scaled)
        
        # 4. Straight-Through Estimator (STE) Trick in JAX
        # Forward pass: Evaluates to z_quantized.
        # Backward pass: jax.lax.stop_gradient blocks the gradient from the 
        # non-differentiable z_quantized, so the gradient flows directly through z_scaled.
        z_ste = z_scaled + jax.lax.stop_gradient(z_quantized - z_scaled)
        
        return z_ste

In [ ]:
!mkdir -p models

In [ ]:
%%writefile models/seer.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Tuple

# Import the FSQ module we defined previously
from models.fsq import FSQ

class Seer(nn.Module):
    """
    The 'Hacker' or Prefrontal Cortex network.
    Observes the global state and generates a discrete compositional message.
    """
    fsq_levels: Sequence[int]
    num_actions: int = 3
    lstm_features: int = 128
    num_message_heads: int = 1

    @nn.compact
    def __call__(
        self, 
        carry: Tuple[jnp.ndarray, jnp.ndarray], 
        map_obs: jnp.ndarray, 
        symbolic_obs: jnp.ndarray,
        target_images: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            map_obs: The Global Map View grid. Expected shape (batch, H, W, C).
            symbolic_obs: Guard Schedule + Sensor States. Expected shape (batch, features).
            target_images: The items the Doers must select. Expected shape (batch, 2, 5, 5, 3).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            discrete_message: The quantized $m_t$ vector sent to the Doer.
            thought_vector: The continuous pre-quantization vector.
            navigation_logits: Action logits used while the Seer is embodied.
        """
        
        # 1. Visual Encoder: CNN for the grid visual 
        x = nn.Conv(features=32, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(map_obs)
        x = nn.relu(x)
        x = nn.Conv(features=64, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(x)
        x = nn.relu(x)
        
        # Flatten visual features to a 1D vector per batch item
        # shape changes from (batch, H, W, channels) to (batch, H * W * channels)
        x_flat = x.reshape((x.shape[0], -1)) 
        
        # 2. Symbolic Encoder: MLP for symbolic data 
        y = nn.Dense(features=64)(symbolic_obs)
        y = nn.relu(y)
        
        # 3. Target Images Encoder
        # target_images shape is (batch, 2, 5, 5, 3)
        ti_flat = target_images.reshape((-1, 5, 5, 3))
        ti_conv1 = nn.Conv(features=16, kernel_size=(3, 3))(ti_flat)
        ti_conv1 = nn.relu(ti_conv1)
        ti_conv2 = nn.Conv(features=32, kernel_size=(3, 3))(ti_conv1)
        ti_conv2 = nn.relu(ti_conv2)
        ti_feats = ti_conv2.reshape((target_images.shape[0], -1))
        
        # 4. Fusion
        # Concatenate the visual, symbolic, and target features
        fused_features = jnp.concatenate([x_flat, y, ti_feats], axis=-1)
        
        # 5. Reasoning Module: LSTM 
        # Evaluates the current state in the context of the previous timestep [cite: 135]
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 5. Continuous Projection
        # Project LSTM hidden state to continuous vector z of size d.
        # Use Orthogonal init with higher scale to prevent FSQ mode collapse.
        thought_vector = nn.Dense(
            features=len(self.fsq_levels) * self.num_message_heads,
            kernel_init=nn.initializers.orthogonal(scale=2.0)
        )(lstm_out)
        thought_vector = thought_vector.reshape(
            (thought_vector.shape[0], self.num_message_heads, len(self.fsq_levels))
        )

        # 6. Output Head: FSQ Discretizer 
        # Transforms the continuous thought vector into the discrete message $m_t$ 
        fsq = FSQ(levels=self.fsq_levels)
        discrete_message = fsq(
            thought_vector.reshape((-1, thought_vector.shape[-1]))
        ).reshape(thought_vector.shape)

        if self.num_message_heads == 1:
            thought_vector = thought_vector[:, 0, :]
            discrete_message = discrete_message[:, 0, :]

        # During the pretraining phase the Seer physically navigates the grid.
        navigation_logits = nn.Dense(features=self.num_actions)(lstm_out)

        return new_carry, discrete_message, thought_vector, navigation_logits

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )


In [ ]:
!mkdir -p models

In [ ]:
%%writefile models/doer.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Tuple

class Doer(nn.Module):
    """
    The 'Thief' or Motor Cortex network.
    Operates on local observations and executes commands via discrete actions.
    """
    fsq_levels: Sequence[int]
    num_actions: int = 9 # e.g., Move N/S/E/W, Toggle, Pick 0/1/2/3
    lstm_features: int = 128
    embed_dim: int = 16

    @nn.compact
    def __call__(
        self,
        carry: Tuple[jnp.ndarray, jnp.ndarray],
        local_obs: jnp.ndarray,
        proprioception: jnp.ndarray,
        message: jnp.ndarray,
        menu_images: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            local_obs: Egocentric 3x3 grid view. Expected shape (batch, 3, 3, C) or zeros.
            proprioception: Internal states (e.g., carrying item). Expected shape (batch, features).
            message: The quantized $m_t$ vector from the Seer. Expected shape (batch, d).
            menu_images: The 4 option images. Expected shape (batch, 4, 5, 5, 3).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            action_logits: Unnormalized log probabilities for the discrete action space.
        """
        
        # 1. Message Encoder: Learned Lookup Table
        # To preserve gradients, we MUST NOT cast to int and use nn.Embed directly 
        # Instead, FSQ naturally provides continuous coordinates (that happen to be quantized).
        # We can just linearly project this entire vector directly into a latent space!
        message_features = nn.Dense(features=self.embed_dim * len(self.fsq_levels))(message)
        message_features = nn.relu(message_features)
        
        # 2. Local Visual Encoder
        # Even for a small 3x3 grid, a single convolution or a dense layer extracts features
        x = nn.Conv(features=16, kernel_size=(2, 2))(local_obs)
        x = nn.relu(x)
        x_flat = x.reshape((x.shape[0], -1))
        
        # 3. Proprioception Encoder
        p = nn.Dense(features=16)(proprioception)
        p = nn.relu(p)
        
        # 4. Menu Images Encoder
        # menu_images shape is (batch, 4, 5, 5, 3)
        mi_flat = menu_images.reshape((-1, 5, 5, 3))
        mi_conv1 = nn.Conv(features=16, kernel_size=(3, 3))(mi_flat)
        mi_conv1 = nn.relu(mi_conv1)
        mi_conv2 = nn.Conv(features=32, kernel_size=(3, 3))(mi_conv1)
        mi_conv2 = nn.relu(mi_conv2)
        mi_feats = mi_conv2.reshape((menu_images.shape[0], -1))
        
        # 5. Fusion
        # Combine local vision, proprioception, menu images, and the embedded command
        fused_features = jnp.concatenate([x_flat, p, mi_feats, message_features], axis=-1)
        
        # 6. Reasoning Module: LSTM
        # Critical for integrating sequences of commands (e.g., maintaining a "Wait" state)
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 6. Output Head: Discrete Action Space
        # Projects the LSTM memory state into logits for physical actions
        action_logits = nn.Dense(features=self.num_actions)(lstm_out)
        
        return new_carry, action_logits

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )

In [ ]:
!mkdir -p agents

In [ ]:
%%writefile agents/mappo.py
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.training.train_state import TrainState
import optax
import chex
from typing import Tuple, Any, Callable
import distrax 
import optax

# 1. Data Structures: Meticulous shape tracking
# Using chex helps enforce accuracy by allowing us to assert shapes later.
@chex.dataclass
class Transition:
    global_obs: chex.Array  # For the Critic (CTDE)
    symbolic_obs: chex.Array # For the Seer
    local_obs: chex.Array   # For the Doer
    proprioception: chex.Array # For the Doer
    message: chex.Array     # For CIC and Heatmap logging
    target_images: chex.Array
    menu_images: chex.Array
    doer_action: chex.Array
    doer_log_prob: chex.Array
    seer_action: chex.Array
    seer_log_prob: chex.Array
    value: chex.Array
    reward: chex.Array
    task_reward: chex.Array
    progress_reward: chex.Array
    follow_reward: chex.Array
    cic_reward_component: chex.Array
    cic_score: chex.Array
    step_penalty_component: chex.Array
    bump_penalty_component: chex.Array
    done: chex.Array
    advantage: chex.Array
    return_val: chex.Array


@chex.dataclass
class TwoDoerTransition:
    global_obs: chex.Array
    symbolic_obs: chex.Array
    local_obs: chex.Array
    proprioception: chex.Array
    message: chex.Array
    target_images: chex.Array
    menu_images: chex.Array
    doer_action: chex.Array
    doer_log_prob: chex.Array
    value: chex.Array
    reward: chex.Array
    task_reward: chex.Array
    progress_reward_per_doer: chex.Array
    step_penalty_component: chex.Array
    wall_penalty_component: chex.Array
    collision_penalty_component: chex.Array
    done: chex.Array
    advantage: chex.Array
    return_val: chex.Array


def _build_message_codebook(message_levels: Tuple[int, ...], dtype: jnp.dtype) -> jnp.ndarray:
    axes = [jnp.arange(level, dtype=dtype) for level in message_levels]
    mesh = jnp.meshgrid(*axes, indexing="ij")
    return jnp.stack(mesh, axis=-1).reshape((-1, len(message_levels)))


def _compute_message_entropy_metrics(
    discrete_messages: chex.Array,
    message_levels: Tuple[int, ...],
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Approximate discrete-code usage with a soft histogram so gradients can flow."""
    flat_messages = discrete_messages.reshape((-1, discrete_messages.shape[-1]))
    codebook = _build_message_codebook(message_levels, flat_messages.dtype)
    sq_distances = jnp.sum(
        jnp.square(flat_messages[:, None, :] - codebook[None, :, :]),
        axis=-1,
    )
    assignment_probs = nn.softmax(-10.0 * sq_distances, axis=-1)
    code_probs = assignment_probs.mean(axis=0)
    code_probs = code_probs / (code_probs.sum() + 1e-8)
    message_entropy = -jnp.sum(code_probs * jnp.log(code_probs + 1e-8))

    num_codes = codebook.shape[0]
    if num_codes > 1:
        normalized_entropy = message_entropy / jnp.log(jnp.asarray(num_codes, dtype=flat_messages.dtype))
    else:
        normalized_entropy = jnp.asarray(0.0, dtype=flat_messages.dtype)

    dominant_code_prob = jnp.max(code_probs)
    return message_entropy, normalized_entropy, dominant_code_prob

# 2. Loss Functions: Separated from network definitions
def calculate_actor_losses(
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    actor_params: dict, # Changed from Any to dict
    transition_batch: Transition,
    init_seer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    control_mode: jnp.ndarray,
    message_levels: Tuple[int, ...],
    clip_eps: float = 0.2,
    entropy_coef: float = 0.01,
    seer_entropy_coef: jnp.ndarray = jnp.array(0.01)
) -> Tuple[jnp.ndarray, dict]:
    """Calculates separate PPO losses for the Seer and Doer reward streams."""
    
    # We must assert shape to prevent silent broadcasting bugs.
    # transition_batch has shape (batch_size, sequence_length, ...) or just (sequence_length, ...)
    # Wait, loop.py returns (num_steps, ...) for single env. Let's assume unbatched or reshape-able.
    # If the user is unrolling over dimension 0:
    
    def scan_fn(carry, transition_step):
        # Step shape assertions:
        # chex.assert_rank(transition_step.action, 1) # (batch_size,)
        # chex.assert_rank(transition_step.global_obs, 4) # (batch_size, H, W, C)
        
        seer_carry, doer_carry = carry
        
        # Seer Forward Pass
        next_seer_carry, discrete_message, thought_vector, seer_nav_logits = seer_apply_fn(
            {"params": actor_params["seer"]},
            seer_carry,
            transition_step.global_obs,
            transition_step.symbolic_obs,
            transition_step.target_images,
        )
        
        # Doer Forward Pass
        next_doer_carry, logits = doer_apply_fn(
            {"params": actor_params["doer"]},
            doer_carry,
            transition_step.local_obs,
            transition_step.proprioception,
            discrete_message,
            transition_step.menu_images
        )
        return (next_seer_carry, next_doer_carry), (
            logits,
            seer_nav_logits,
            discrete_message,
            thought_vector,
        )
        
    _, (doer_logits, seer_nav_logits, discrete_messages, thought_vectors) = jax.lax.scan(
        scan_fn, 
        (init_seer_carry, init_doer_carry), 
        transition_batch
    )

    communication_mode = control_mode == 1
    doer_pi = distrax.Categorical(logits=doer_logits)
    seer_pi = distrax.Categorical(logits=seer_nav_logits)
    doer_new_log_probs = doer_pi.log_prob(transition_batch.doer_action)
    seer_nav_new_log_probs = seer_pi.log_prob(transition_batch.seer_action)
    doer_entropy = doer_pi.entropy().mean()
    seer_nav_entropy = seer_pi.entropy().mean()

    seer_adv = transition_batch.advantage[..., 0]
    doer_adv = transition_batch.advantage[..., 1]
    seer_adv = (seer_adv - seer_adv.mean()) / (seer_adv.std() + 1e-8)
    doer_adv = (doer_adv - doer_adv.mean()) / (doer_adv.std() + 1e-8)

    seer_old_log_probs = jnp.where(
        communication_mode,
        transition_batch.doer_log_prob,
        transition_batch.seer_log_prob,
    )
    seer_new_log_probs = jnp.where(
        communication_mode,
        doer_new_log_probs,
        seer_nav_new_log_probs,
    )
    seer_logratio = seer_new_log_probs - seer_old_log_probs
    seer_ratio = jnp.exp(seer_logratio)

    seer_loss_unclipped = seer_ratio * seer_adv
    seer_loss_clipped = jnp.clip(seer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * seer_adv
    seer_actor_loss = -jnp.minimum(seer_loss_unclipped, seer_loss_clipped).mean()

    doer_logratio = doer_new_log_probs - transition_batch.doer_log_prob
    doer_ratio = jnp.exp(doer_logratio)
    doer_loss_unclipped = doer_ratio * doer_adv
    doer_loss_clipped = jnp.clip(doer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * doer_adv
    doer_actor_loss = -jnp.minimum(doer_loss_unclipped, doer_loss_clipped).mean()
    doer_actor_loss = jnp.where(communication_mode, doer_actor_loss, 0.0)

    message_entropy, message_entropy_normalized, dominant_code_prob = (
        _compute_message_entropy_metrics(discrete_messages, message_levels)
    )
    seer_bonus = jnp.where(communication_mode, message_entropy, seer_nav_entropy)

    seer_loss = seer_actor_loss - seer_entropy_coef * seer_bonus
    doer_loss = doer_actor_loss - jnp.where(communication_mode, entropy_coef * doer_entropy, 0.0)

    return (seer_loss, doer_loss), {
        "seer_actor_loss": seer_actor_loss,
        "doer_actor_loss": doer_actor_loss,
        "entropy": jnp.where(communication_mode, doer_entropy, seer_nav_entropy),
        "seer_ratio": seer_ratio.mean(),
        "doer_ratio": doer_ratio.mean(),
        "message_entropy": message_entropy,
        "message_entropy_normalized": message_entropy_normalized,
        "message_dominant_probability": dominant_code_prob,
        "seer_nav_entropy": seer_nav_entropy,
        "discrete_messages": discrete_messages
    }

def calculate_critic_loss(
    critic_apply_fn: Callable,
    critic_params: Any,
    transition_batch: Transition,
    value_clip: float = 0.2
) -> Tuple[jnp.ndarray, dict]:
    """Calculates the value loss for the centralized critic."""
    
    # Assert shape
    chex.assert_rank(transition_batch.global_obs, 4) # (batch_size, H, W, C) since it is flattened over sequence
    
    # The critic uses the global observation (CTDE)
    values = critic_apply_fn({"params": critic_params}, transition_batch.global_obs)

    value_pred_clipped = transition_batch.value + jnp.clip(
        values - transition_batch.value, -value_clip, value_clip
    )
    
    value_losses = jnp.square(values - transition_batch.return_val)
    value_losses_clipped = jnp.square(value_pred_clipped - transition_batch.return_val)
    
    critic_loss = 0.5 * jnp.maximum(value_losses, value_losses_clipped).mean()
    
    return critic_loss, {"critic_loss": critic_loss}


def calculate_two_doer_actor_losses(
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    actor_params: dict,
    transition_batch: TwoDoerTransition,
    init_seer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    message_levels: Tuple[int, ...],
    clip_eps: float = 0.2,
    entropy_coef: float = 0.01,
    seer_entropy_coef: jnp.ndarray = jnp.array(0.01),
) -> Tuple[jnp.ndarray, dict]:
    """PPO actor losses for one Seer coordinating two embodied Doers."""

    def scan_fn(carry, transition_step):
        seer_carry, doer_carry = carry
        next_seer_carry, discrete_messages, _, _ = seer_apply_fn(
            {"params": actor_params["seer"]},
            seer_carry,
            transition_step.global_obs,
            transition_step.symbolic_obs,
            transition_step.target_images,
        )
        batch_size, num_doers = transition_step.local_obs.shape[:2]
        flat_local_obs = transition_step.local_obs.reshape(
            (batch_size * num_doers,) + transition_step.local_obs.shape[2:]
        )
        flat_proprioception = transition_step.proprioception.reshape(
            (batch_size * num_doers,) + transition_step.proprioception.shape[2:]
        )
        flat_messages = discrete_messages.reshape(
            (batch_size * num_doers,) + discrete_messages.shape[2:]
        )
        flat_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_doer_carry, flat_logits = doer_apply_fn(
            {"params": actor_params["doer"]},
            flat_doer_carry,
            flat_local_obs,
            flat_proprioception,
            flat_messages,
            transition_step.menu_images.reshape((batch_size * num_doers,) + transition_step.menu_images.shape[2:])
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_doer_carry,
        )
        doer_logits = flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))
        return (next_seer_carry, next_doer_carry), (doer_logits, discrete_messages)

    _, (doer_logits, discrete_messages) = jax.lax.scan(
        scan_fn,
        (init_seer_carry, init_doer_carry),
        transition_batch,
    )

    doer_pi = distrax.Categorical(logits=doer_logits)
    doer_new_log_probs = doer_pi.log_prob(transition_batch.doer_action)
    doer_entropy = doer_pi.entropy().mean()

    team_adv = transition_batch.advantage
    team_adv = (team_adv - team_adv.mean()) / (team_adv.std() + 1e-8)
    doer_old_log_probs = transition_batch.doer_log_prob

    seer_old_log_probs = doer_old_log_probs.sum(axis=-1)
    seer_new_log_probs = doer_new_log_probs.sum(axis=-1)
    seer_logratio = seer_new_log_probs - seer_old_log_probs
    seer_ratio = jnp.exp(seer_logratio)
    seer_loss_unclipped = seer_ratio * team_adv
    seer_loss_clipped = jnp.clip(seer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * team_adv
    seer_actor_loss = -jnp.minimum(seer_loss_unclipped, seer_loss_clipped).mean()

    doer_logratio = doer_new_log_probs - doer_old_log_probs
    doer_ratio = jnp.exp(doer_logratio)
    team_adv_expanded = team_adv[..., None]
    doer_loss_unclipped = doer_ratio * team_adv_expanded
    doer_loss_clipped = (
        jnp.clip(doer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * team_adv_expanded
    )
    doer_actor_loss = -jnp.minimum(doer_loss_unclipped, doer_loss_clipped).mean()

    message_entropy, message_entropy_normalized, dominant_code_prob = (
        _compute_message_entropy_metrics(discrete_messages, message_levels)
    )
    seer_loss = seer_actor_loss - seer_entropy_coef * message_entropy
    doer_loss = doer_actor_loss - entropy_coef * doer_entropy

    return (seer_loss, doer_loss), {
        "seer_actor_loss": seer_actor_loss,
        "doer_actor_loss": doer_actor_loss,
        "entropy": doer_entropy,
        "seer_ratio": seer_ratio.mean(),
        "doer_ratio": doer_ratio.mean(),
        "message_entropy": message_entropy,
        "message_entropy_normalized": message_entropy_normalized,
        "message_dominant_probability": dominant_code_prob,
        "discrete_messages": discrete_messages,
    }


def calculate_two_doer_critic_loss(
    critic_apply_fn: Callable,
    critic_params: Any,
    transition_batch: TwoDoerTransition,
    value_clip: float = 0.2,
) -> Tuple[jnp.ndarray, dict]:
    values = critic_apply_fn({"params": critic_params}, transition_batch.global_obs).squeeze(-1)
    value_pred_clipped = transition_batch.value + jnp.clip(
        values - transition_batch.value,
        -value_clip,
        value_clip,
    )
    value_losses = jnp.square(values - transition_batch.return_val)
    value_losses_clipped = jnp.square(value_pred_clipped - transition_batch.return_val)
    critic_loss = 0.5 * jnp.maximum(value_losses, value_losses_clipped).mean()
    return critic_loss, {"critic_loss": critic_loss}

# 3. The Update Step: JIT-compiled gradient application
import functools

@functools.partial(
    jax.jit,
    static_argnames=(
        "seer_apply_fn",
        "doer_apply_fn",
        "message_levels",
        "num_ppo_epochs",
        "num_minibatches",
    ),
)
def update_actor(
    actor_state: TrainState, 
    transition_batch: Transition,
    init_seer_carry: Any,
    init_doer_carry: Any,
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    control_mode: jnp.ndarray,
    message_levels: Tuple[int, ...],
    seer_entropy_coef: jnp.ndarray,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the actor network using PPO epochs."""
    
    # Add a batch dimension if missing (assumes trajectory is num_steps, ...)
    # Wait, the prompt says trajectory is (batch_size, seq_len, ...)
    # If the user scales num_envs later, batch_size = num_envs
    
    batch_size = transition_batch.doer_action.shape[0]
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        actor_state, key = carry
        key, subkey = jax.random.split(key)
        
        # Shuffle along the batch dimension
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx):
            # Slice the minibatch
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], transition_batch)
            
            # Since calculate_actor_loss currently assumes scan over time, and time is dim 1:
            # wait, if input is (batch, time, ...) and scan is over time, we must swap axes!
            # scan_fn expects transition sequence to be the leading dimension.
            # So let's swap seq_len (dim 1) to be dim 0 for scan.
            mb_transition_time_first = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 0, 1), mb_transition)
            
            # Extract initial carries for this minibatch
            # Assuming init_seer_carry is (batch_size, ...)
            mb_seer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_seer_carry)
            mb_doer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_doer_carry)
            
            def seer_loss_fn(seer_params):
                (seer_loss, _), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": seer_params, "doer": state.params["doer"]},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    control_mode=control_mode,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return seer_loss, metrics

            def doer_loss_fn(doer_params):
                (_, doer_loss), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": state.params["seer"], "doer": doer_params},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    control_mode=control_mode,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return doer_loss, metrics

            (seer_loss, seer_metrics), seer_grads = jax.value_and_grad(
                seer_loss_fn,
                has_aux=True,
            )(state.params["seer"])
            (doer_loss, doer_metrics), doer_grads = jax.value_and_grad(
                doer_loss_fn,
                has_aux=True,
            )(state.params["doer"])

            grads = {"seer": seer_grads, "doer": doer_grads}

            # Record explicit gradient norms for auditing
            seer_grad_norm = optax.global_norm(grads["seer"])
            doer_grad_norm = optax.global_norm(grads["doer"])

            metrics = {
                "seer_loss": seer_loss,
                "doer_loss": doer_loss,
                "seer_actor_loss": seer_metrics["seer_actor_loss"],
                "doer_actor_loss": doer_metrics["doer_actor_loss"],
                "entropy": doer_metrics["entropy"],
                "seer_ratio": seer_metrics["seer_ratio"],
                "doer_ratio": doer_metrics["doer_ratio"],
                "message_entropy": seer_metrics["message_entropy"],
                "message_entropy_normalized": seer_metrics["message_entropy_normalized"],
                "message_dominant_probability": seer_metrics["message_dominant_probability"],
                "seer_nav_entropy": seer_metrics["seer_nav_entropy"],
                "discrete_messages": seer_metrics["discrete_messages"],
                "seer_grad_norm": seer_grad_norm,
                "doer_grad_norm": doer_grad_norm,
                "actor_loss": seer_loss + doer_loss,
            }
            
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        # Minibatch loop (scan over start_indices)
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        actor_state, mb_metrics = jax.lax.scan(minibatch_fn, actor_state, start_indices)
        
        # Average metrics over minibatches
        epoch_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in mb_metrics.items()}
        return (actor_state, key), epoch_metrics
        
    (final_actor_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (actor_state, rng), None, length=num_ppo_epochs
    )
    
    # Return averaged metrics over epochs
    final_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in epoch_metrics.items()}
    return final_actor_state, final_metrics

@functools.partial(jax.jit, static_argnames=("critic_apply_fn", "num_ppo_epochs", "num_minibatches"))
def update_critic(
    critic_state: TrainState, 
    transition_batch: Transition,
    critic_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the critic network."""
    
    # For critic we can flatten the batch and sequence dimension since it's just MLPs
    batch_size = transition_batch.doer_action.shape[0] * transition_batch.doer_action.shape[1]
    flat_transition = jax.tree_util.tree_map(lambda x: x.reshape((batch_size,) + x.shape[2:]), transition_batch)
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        critic_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], flat_transition)
            
            loss_fn = lambda params: calculate_critic_loss(
                critic_apply_fn, params, mb_transition
            )
            
            (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        critic_state, mb_metrics = jax.lax.scan(minibatch_fn, critic_state, start_indices)
        
        epoch_metrics = jax.tree_util.tree_map(lambda x: x.mean(), mb_metrics)
        return (critic_state, key), epoch_metrics

    (final_critic_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (critic_state, rng), None, length=num_ppo_epochs
    )
    
    final_metrics = jax.tree_util.tree_map(lambda x: x.mean(), epoch_metrics)
    return final_critic_state, final_metrics


@functools.partial(
    jax.jit,
    static_argnames=(
        "seer_apply_fn",
        "doer_apply_fn",
        "message_levels",
        "num_ppo_epochs",
        "num_minibatches",
    ),
)
def update_actor_two_doer(
    actor_state: TrainState,
    transition_batch: TwoDoerTransition,
    init_seer_carry: Any,
    init_doer_carry: Any,
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    message_levels: Tuple[int, ...],
    seer_entropy_coef: jnp.ndarray,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1,
) -> Tuple[TrainState, dict]:
    batch_size = transition_batch.doer_action.shape[0]
    minibatch_size = batch_size // num_minibatches

    def epoch_fn(carry, _):
        actor_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)

        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], transition_batch)
            mb_transition_time_first = jax.tree_util.tree_map(
                lambda x: jnp.swapaxes(x, 0, 1),
                mb_transition,
            )
            mb_seer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_seer_carry)
            mb_doer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_doer_carry)

            def seer_loss_fn(seer_params):
                (seer_loss, _), metrics = calculate_two_doer_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": seer_params, "doer": state.params["doer"]},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return seer_loss, metrics

            def doer_loss_fn(doer_params):
                (_, doer_loss), metrics = calculate_two_doer_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": state.params["seer"], "doer": doer_params},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return doer_loss, metrics

            (seer_loss, seer_metrics), seer_grads = jax.value_and_grad(
                seer_loss_fn,
                has_aux=True,
            )(state.params["seer"])
            (doer_loss, doer_metrics), doer_grads = jax.value_and_grad(
                doer_loss_fn,
                has_aux=True,
            )(state.params["doer"])
            grads = {"seer": seer_grads, "doer": doer_grads}
            seer_grad_norm = optax.global_norm(grads["seer"])
            doer_grad_norm = optax.global_norm(grads["doer"])
            metrics = {
                "seer_loss": seer_loss,
                "doer_loss": doer_loss,
                "seer_actor_loss": seer_metrics["seer_actor_loss"],
                "doer_actor_loss": doer_metrics["doer_actor_loss"],
                "entropy": doer_metrics["entropy"],
                "seer_ratio": seer_metrics["seer_ratio"],
                "doer_ratio": doer_metrics["doer_ratio"],
                "message_entropy": seer_metrics["message_entropy"],
                "message_entropy_normalized": seer_metrics["message_entropy_normalized"],
                "message_dominant_probability": seer_metrics["message_dominant_probability"],
                "discrete_messages": seer_metrics["discrete_messages"],
                "seer_grad_norm": seer_grad_norm,
                "doer_grad_norm": doer_grad_norm,
                "actor_loss": seer_loss + doer_loss,
            }
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics

        start_indices = jnp.arange(0, batch_size, minibatch_size)
        actor_state, mb_metrics = jax.lax.scan(minibatch_fn, actor_state, start_indices)
        epoch_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in mb_metrics.items()}
        return (actor_state, key), epoch_metrics

    (final_actor_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn,
        (actor_state, rng),
        None,
        length=num_ppo_epochs,
    )
    final_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in epoch_metrics.items()}
    return final_actor_state, final_metrics


@functools.partial(jax.jit, static_argnames=("critic_apply_fn", "num_ppo_epochs", "num_minibatches"))
def update_critic_two_doer(
    critic_state: TrainState,
    transition_batch: TwoDoerTransition,
    critic_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1,
) -> Tuple[TrainState, dict]:
    batch_size = transition_batch.doer_action.shape[0] * transition_batch.doer_action.shape[1]
    flat_transition = jax.tree_util.tree_map(
        lambda x: x.reshape((batch_size,) + x.shape[2:]),
        transition_batch,
    )
    minibatch_size = batch_size // num_minibatches

    def epoch_fn(carry, _):
        critic_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)

        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], flat_transition)
            loss_fn = lambda params: calculate_two_doer_critic_loss(
                critic_apply_fn,
                params,
                mb_transition,
            )
            (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics

        start_indices = jnp.arange(0, batch_size, minibatch_size)
        critic_state, mb_metrics = jax.lax.scan(minibatch_fn, critic_state, start_indices)
        epoch_metrics = jax.tree_util.tree_map(lambda x: x.mean(), mb_metrics)
        return (critic_state, key), epoch_metrics

    (final_critic_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn,
        (critic_state, rng),
        None,
        length=num_ppo_epochs,
    )
    final_metrics = jax.tree_util.tree_map(lambda x: x.mean(), epoch_metrics)
    return final_critic_state, final_metrics


In [ ]:
!mkdir -p training

In [ ]:
%%writefile training/gae.py
import jax
import jax.numpy as jnp
from typing import Tuple

def compute_gae(
    rewards: jnp.ndarray,
    values: jnp.ndarray,
    dones: jnp.ndarray,
    last_val: jnp.ndarray,
    gamma: float = 0.99,
    gae_lambda: float = 0.95
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    """
    Computes Generalized Advantage Estimation (GAE) and target returns.
    
    Args:
        rewards: Array of rewards collected during the rollout. Shape: (num_steps,)
        values: Array of value predictions from the Critic. Shape: (num_steps,)
        dones: Array of boolean/integer done flags. Shape: (num_steps,)
        last_val: The Critic's value prediction for the state *after* the final step.
        gamma: Discount factor.
        gae_lambda: Bias-variance tradeoff parameter for GAE.
        
    Returns:
        advantages: The calculated GAE advantages. Shape: (num_steps,)
        returns: The target values for the Critic to learn from (Advantages + Values).
    """
    
    def _gae_step(carry, transition_data):
        """A single step of the reverse scan."""
        gae_t_plus_1 = carry
        reward, value, done, next_value = transition_data
        
        # Calculate the Temporal Difference (TD) error
        # If the episode ended (done=1), the next state has no value.
        delta = reward + gamma * next_value * (1.0 - done) - value
        
        # Calculate the advantage for the current timestep
        gae_t = delta + gamma * gae_lambda * (1.0 - done) * gae_t_plus_1
        
        # Pass the current advantage back as the carry for the previous timestep
        return gae_t, gae_t

    # To calculate the TD error, we need the value of the next state for every step.
    # We create an array of "next values" by shifting the values array by one 
    # and appending the bootstrap 'last_val' at the end.
    next_values = jnp.append(values[1:], last_val)
    
    # Pack the data for the scan
    scan_data = (rewards, values, dones, next_values)
    
    # Initialize the carry with 0.0 (the advantage after the final step)
    initial_gae = 0.0
    
    # Run the scan in reverse to propagate advantages backwards
    _, advantages = jax.lax.scan(_gae_step, initial_gae, scan_data, reverse=True)
    
    # The return value (target for the critic) is simply the advantage + the predicted value
    returns = advantages + values
    
    return advantages, returns

In [ ]:
!mkdir -p training

In [ ]:
%%writefile training/population.py


In [ ]:
!mkdir -p training

In [ ]:
%%writefile training/loop.py
import jax
import jax.numpy as jnp
import distrax
from typing import Tuple, Any, Dict
from training.gae import compute_gae 

# Assuming Transition is imported from your mappo.py or a shared datatypes file
from agents.mappo import Transition, TwoDoerTransition
from eval.metrics import compute_cic

def make_rollout_step(
    env,
    seer_apply_fn,
    doer_apply_fn,
    critic_apply_fn,
    follow_reward_scale=0.1,
):
    """
    A closure that returns the JAX-compilable step function.
    Passing the environment and apply functions here avoids passing them 
    repeatedly into the compiled loop.
    """
    follow_reward_scale = jnp.asarray(follow_reward_scale, dtype=jnp.float32)
    
    def rollout_step(runner_state: Tuple, _):
        """
        Executes a single environment step and network forward pass.
        Designed to be passed directly to jax.lax.scan.
        """
        # Unpack the runner state
        (
            params,
            seer_carry,
            doer_carry,
            env_state,
            env_obs,
            rng,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        ) = runner_state
        num_envs = env_obs["global_map"].shape[0]
        communication_mode = control_mode == 1
        
        # Split the PRNG key for the stochastic actions
        rng, seer_rng, doer_rng, env_rng = jax.random.split(rng, 4)
        env_step_keys = jax.random.split(env_rng, num_envs)
        
        # 1. Seer Forward Pass (Prefrontal Cortex)
        # Enforcing CTDE: Seer gets the global view[cite: 131, 132].
        # In a custom jaxmarl wrapper, you would extract these from env_obs
        global_map = env_obs["global_map"]
        symbolic_obs = env_obs["symbolic_state"]
        
        next_seer_carry, discrete_message, _, seer_nav_logits = seer_apply_fn(
            {"params": params["seer"]}, 
            seer_carry, 
            global_map, 
            symbolic_obs,
            env_obs["target_images"]
        )
        
        # 2. Doer Forward Pass (Motor Cortex)
        # Enforcing Functional Asymmetry: Doer gets local view and the message[cite: 137, 138].
        local_obs = env_obs["local_view"]
        proprioception = env_obs["proprioception"]
        
        next_doer_carry, action_logits = doer_apply_fn(
            {"params": params["doer"]}, 
            doer_carry, 
            local_obs, 
            proprioception, 
            discrete_message,
            env_obs["menu_images"]
        )
        _, null_action_logits = doer_apply_fn(
            {"params": params["doer"]},
            doer_carry,
            local_obs,
            proprioception,
            jnp.zeros_like(discrete_message),
            env_obs["menu_images"]
        )
        
        # 3. Action Selection
        seer_pi = distrax.Categorical(logits=seer_nav_logits)
        pi = distrax.Categorical(logits=action_logits)
        seer_action = seer_pi.sample(seed=seer_rng)
        doer_action = pi.sample(seed=doer_rng)
        seer_log_prob = seer_pi.log_prob(seer_action)
        doer_log_prob = pi.log_prob(doer_action)
        env_action = jnp.where(communication_mode, doer_action, seer_action)
        message_action = jnp.argmax(action_logits, axis=-1)
        null_message_action = jnp.argmax(null_action_logits, axis=-1)
        action_change_bonus = (
            message_action != null_message_action
        ).astype(jnp.float32) * follow_reward_scale
        
        # 4. Critic Forward Pass (Centralized Training)
        # The critic evaluates the global state to guide learning[cite: 111].
        value = critic_apply_fn({"params": params["critic"]}, global_map)
        
        # 5. Environment Step
        # Step the jaxmarl environment using the chosen action
        # Note: jaxmarl expects a dictionary of actions for multi-agent, 
        # adapt this based on your specific wrapper implementation.
        next_env_obs, next_env_state, reward, done, info = env.step_batch(
            env_step_keys,
            env_state,
            env_action,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )

        task_reward = info["task_reward"]
        progress_reward = info["progress_reward"]
        step_penalty = info["step_penalty"]
        bump_penalty = info["bump_penalty"]
        useful_communication = jnp.logical_or(
            progress_reward > 0.0,
            task_reward > 0.0,
        )
        follow_reward = jnp.where(
            jnp.logical_and(communication_mode, useful_communication),
            action_change_bonus,
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        shared_comm_reward = (
            task_reward + progress_reward + follow_reward - step_penalty - bump_penalty
        )
        seer_reward = jnp.where(
            communication_mode,
            shared_comm_reward,
            task_reward + progress_reward - step_penalty - bump_penalty,
        )
        doer_reward = jnp.where(
            communication_mode,
            shared_comm_reward,
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        reward = jnp.stack([seer_reward, doer_reward], axis=-1)

        done_mask = done[:, None]
        next_seer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_seer_carry,
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_doer_carry,
        )
        
        # 6. Build the Transition
        transition = Transition(
            global_obs=global_map,
            symbolic_obs=symbolic_obs,
            local_obs=local_obs,
            proprioception=proprioception,
            message=discrete_message,
            target_images=env_obs["target_images"],
            menu_images=env_obs["menu_images"],
            doer_action=doer_action,
            doer_log_prob=doer_log_prob,
            seer_action=seer_action,
            seer_log_prob=seer_log_prob,
            value=value,
            reward=reward,
            task_reward=task_reward,
            progress_reward=progress_reward,
            follow_reward=follow_reward,
            cic_reward_component=jnp.zeros_like(task_reward),
            cic_score=jnp.zeros_like(task_reward),
            step_penalty_component=step_penalty,
            bump_penalty_component=bump_penalty,
            done=done,
            # Advantage and return will be calculated post-rollout using GAE
            advantage=jnp.zeros_like(reward), 
            return_val=jnp.zeros_like(reward)
        )
        
        # Repack the updated runner state
        next_runner_state = (
            params,
            next_seer_carry,
            next_doer_carry,
            next_env_state,
            next_env_obs,
            rng,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        )
        
        return next_runner_state, transition

    return rollout_step

import functools

@functools.partial(jax.jit, static_argnames=("num_steps", "step_fn", "critic_apply_fn", "doer_apply_fn"))
def generate_trajectory_and_gae(
    params, rng, env_obs, env_state, seer_carry, doer_carry, vision_radius: jnp.ndarray, control_mode: jnp.ndarray, fixed_goal_position: jnp.ndarray, fixed_start_position: jnp.ndarray, cic_coef: jnp.ndarray, num_steps: int,
    step_fn, critic_apply_fn, doer_apply_fn
):
    """
    Executes the full episode rollout and computes GAE in a single compiled pass.
    Note: We pass the pre-compiled step_fn and initial states directly here 
    for better JAX compilation efficiency.
    """
    initial_runner_state = (
        params,
        seer_carry,
        doer_carry,
        env_state,
        env_obs,
        rng,
        vision_radius,
        control_mode,
        fixed_goal_position,
        fixed_start_position,
    )
    
    # 1. Execute the scan loop to collect the raw trajectory
    final_runner_state, trajectory_batch = jax.lax.scan(
        step_fn, initial_runner_state, None, length=num_steps
    )
    
    # 2. Extract the final state for Critic bootstrapping
    # Unpack the final runner state to get the last env_obs
    _, _, _, _, final_env_obs, final_rng, _, _, _, _ = final_runner_state
    
    # Enforce CTDE: The critic evaluates the global map 
    final_global_map = final_env_obs["global_map"]
    
    # 3. Calculate the bootstrap value (last_val)
    # The critic evaluates the state *after* the final step
    last_val = critic_apply_fn({"params": params["critic"]}, final_global_map)
    
    communication_mode = control_mode == 1

    def add_cic_bonus(_):
        cic_score = compute_cic(
            doer_apply_fn,
            params["doer"],
            trajectory_batch,
            doer_carry,
            final_rng,
        )
        per_step_cic_reward = cic_coef * cic_score / jnp.asarray(num_steps, dtype=jnp.float32)
        cic_reward_component = jnp.full_like(trajectory_batch.task_reward, per_step_cic_reward)
        cic_score_component = jnp.full_like(trajectory_batch.task_reward, cic_score)
        reward_with_cic = trajectory_batch.reward + cic_reward_component[..., None]
        return reward_with_cic, cic_reward_component, cic_score_component

    def skip_cic_bonus(_):
        zeros = jnp.zeros_like(trajectory_batch.task_reward)
        return trajectory_batch.reward, zeros, zeros

    reward_with_cic, cic_reward_component, cic_score_component = jax.lax.cond(
        jnp.logical_and(communication_mode, cic_coef > 0.0),
        add_cic_bonus,
        skip_cic_bonus,
        operand=None,
    )
    
    # 5. Compute GAE
    # Note: If you scale up to multiple environments (num_envs > 1), you would wrap 
    # compute_gae with jax.vmap(compute_gae, in_axes=1, out_axes=1) to vectorize 
    # the advantage calculation across all environments simultaneously.
    advantages, returns = jax.vmap(
        jax.vmap(
            compute_gae,
            in_axes=(1, 1, 1, 0, None, None),
            out_axes=1,
        ),
        in_axes=(2, 2, None, 1, None, None),
        out_axes=2,
    )(
        reward_with_cic,
        trajectory_batch.value,
        trajectory_batch.done,
        last_val,
        0.99,
        0.95,
    )
    
    # 5. Update the trajectory batch
    # Using the .replace() method provided by chex/flax dataclasses
    trajectory_batch = trajectory_batch.replace(
        reward=reward_with_cic,
        cic_reward_component=cic_reward_component,
        cic_score=cic_score_component,
        advantage=advantages,
        return_val=returns
    )
    
    return final_runner_state, trajectory_batch


def make_two_doer_rollout_step(
    env,
    seer_apply_fn,
    doer_apply_fn,
    critic_apply_fn,
):
    """Rollout step for one Seer coordinating two embodied Doers."""

    def rollout_step(runner_state: Tuple, _):
        params, seer_carry, doer_carry, env_state, env_obs, rng, fixed_positions = runner_state
        num_envs = env_obs["global_map"].shape[0]
        global_map = env_obs["global_map"]
        symbolic_obs = env_obs["symbolic_state"]
        local_obs = env_obs["local_views"]
        proprioception = env_obs["proprioceptions"]

        rng, action_rng, env_rng = jax.random.split(rng, 3)
        env_step_keys = jax.random.split(env_rng, num_envs)

        next_seer_carry, discrete_messages, _, _ = seer_apply_fn(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_obs,
            env_obs["target_images"]
        )

        batch_size, num_doers = local_obs.shape[:2]
        flat_local_obs = local_obs.reshape((batch_size * num_doers,) + local_obs.shape[2:])
        flat_proprioception = proprioception.reshape(
            (batch_size * num_doers,) + proprioception.shape[2:]
        )
        flat_messages = discrete_messages.reshape(
            (batch_size * num_doers,) + discrete_messages.shape[2:]
        )
        flat_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_doer_carry, flat_logits = doer_apply_fn(
            {"params": params["doer"]},
            flat_doer_carry,
            flat_local_obs,
            flat_proprioception,
            flat_messages,
            env_obs["menu_images"].reshape((batch_size * num_doers,) + env_obs["menu_images"].shape[2:])
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_doer_carry,
        )
        doer_logits = flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))
        doer_pi = distrax.Categorical(logits=doer_logits)
        doer_action = doer_pi.sample(seed=action_rng)
        doer_log_prob = doer_pi.log_prob(doer_action)
        value = critic_apply_fn({"params": params["critic"]}, global_map).squeeze(-1)

        next_env_obs, next_env_state, reward, done, info = env.step_batch(
            env_step_keys,
            env_state,
            doer_action,
            fixed_positions=fixed_positions,
        )

        next_seer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done[:, None], jnp.zeros_like(x), x),
            next_seer_carry,
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done[:, None, None], jnp.zeros_like(x), x),
            next_doer_carry,
        )

        transition = TwoDoerTransition(
            global_obs=global_map,
            symbolic_obs=symbolic_obs,
            local_obs=local_obs,
            proprioception=proprioception,
            message=discrete_messages,
            target_images=env_obs["target_images"],
            menu_images=env_obs["menu_images"],
            doer_action=doer_action,
            doer_log_prob=doer_log_prob,
            value=value,
            reward=reward,
            task_reward=info["task_reward"],
            progress_reward_per_doer=info["progress_reward_per_doer"],
            step_penalty_component=info["step_penalty"],
            wall_penalty_component=info["wall_penalty"],
            collision_penalty_component=info["collision_penalty"],
            done=done,
            advantage=jnp.zeros_like(reward),
            return_val=jnp.zeros_like(reward),
        )

        next_runner_state = (
            params,
            next_seer_carry,
            next_doer_carry,
            next_env_state,
            next_env_obs,
            rng,
            fixed_positions,
        )
        return next_runner_state, transition

    return rollout_step


@functools.partial(
    jax.jit,
    static_argnames=("num_steps", "step_fn", "critic_apply_fn"),
)
def generate_two_doer_trajectory_and_gae(
    params,
    rng,
    env_obs,
    env_state,
    seer_carry,
    doer_carry,
    fixed_positions,
    num_steps: int,
    step_fn,
    critic_apply_fn,
):
    initial_runner_state = (
        params,
        seer_carry,
        doer_carry,
        env_state,
        env_obs,
        rng,
        fixed_positions,
    )
    final_runner_state, trajectory_batch = jax.lax.scan(
        step_fn,
        initial_runner_state,
        None,
        length=num_steps,
    )
    _, _, _, _, final_env_obs, _, _ = final_runner_state
    last_val = critic_apply_fn(
        {"params": params["critic"]},
        final_env_obs["global_map"],
    ).squeeze(-1)
    advantages, returns = jax.vmap(
        compute_gae,
        in_axes=(1, 1, 1, 0, None, None),
        out_axes=1,
    )(
        trajectory_batch.reward,
        trajectory_batch.value,
        trajectory_batch.done,
        last_val,
        0.99,
        0.95,
    )
    trajectory_batch = trajectory_batch.replace(
        advantage=advantages,
        return_val=returns,
    )
    return final_runner_state, trajectory_batch


In [ ]:
!mkdir -p eval

In [ ]:
%%writefile eval/visualize.py
from pathlib import Path

import jax
import jax.numpy as jnp
from PIL import Image

from envs.navix_wrapper import UNSET_POSITION


def visualize_episode(
    env,
    params,
    rng,
    seer,
    doer,
    filename="episode.gif",
    vision_radius=jnp.array(2.0),
    max_steps=200,
    control_mode=jnp.array(1, dtype=jnp.int32),
    fixed_goal_position=UNSET_POSITION,
    fixed_start_position=UNSET_POSITION,
):
    """Run one greedy evaluation episode and save it as a GIF."""
    frames = []
    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(
        reset_rng,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)

    done = False
    solved = False
    step_count = 0

    while not bool(done) and step_count < max_steps:
        frame = env.render(state, control_mode=int(control_mode))
        frames.append(Image.fromarray(frame))

        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_view = obs["local_view"][None, ...]
        proprioception = obs["proprioception"][None, ...]

        seer_carry, message, _, seer_nav_logits = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
        )

        if int(control_mode) == env.SEER_NAV_PHASE:
            action = jnp.argmax(seer_nav_logits[0]).astype(jnp.int32)
        else:
            doer_carry, action_logits = doer.apply(
                {"params": params["doer"]},
                doer_carry,
                local_view,
                proprioception,
                message,
            )
            action = jnp.argmax(action_logits[0]).astype(jnp.int32)

        rng, step_rng = jax.random.split(rng)
        obs, state, reward, done, info = env.step(
            step_rng,
            state,
            action,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )
        solved = solved or bool(done) and float(info["task_reward"]) > 0.0

        step_count += 1

    if not frames:
        raise RuntimeError("Visualization episode produced no frames.")

    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        duration=120,
        loop=0,
    )
    print(f"Episode saved to {output_path}")
    return output_path, solved


In [ ]:
!mkdir -p eval

In [ ]:
%%writefile eval/metrics.py
import jax
import jax.numpy as jnp
from typing import Tuple, Callable
import functools

@functools.partial(jax.jit, static_argnames=("doer_apply_fn",))
def compute_cic(
    doer_apply_fn: Callable,
    doer_params: dict,
    transition_batch, 
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    rng: jax.random.PRNGKey
) -> jnp.ndarray:
    """
    Computes Causal Influence of Communication (CIC) by ablating the Seer's message
    and measuring the divergence in the Doer's resulting deterministic actions.
    """
    
    def scan_fn(carry, step_data):
        doer_carry = carry
        msg, loc, prop, menu = step_data
        
        next_doer_carry, logits = doer_apply_fn(
            {"params": doer_params},
            doer_carry,
            loc,
            prop,
            msg,
            menu
        )
        return next_doer_carry, logits

    # Transition batch shapes: (seq_len, num_envs, ...) from loop.py
    loc = transition_batch.local_obs
    prop = transition_batch.proprioception
    msg_true = transition_batch.message
    menu = transition_batch.menu_images

    # True forward pass
    _, true_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_true, loc, prop, menu))
    
    # Ablated forward pass: shuffle messages independently over time for each env.
    env_keys = jax.random.split(rng, msg_true.shape[1])
    msg_shuffled = jax.vmap(
        lambda key, msgs: jax.random.permutation(key, msgs, axis=0)
    )(env_keys, jnp.swapaxes(msg_true, 0, 1))
    msg_shuffled = jnp.swapaxes(msg_shuffled, 0, 1)
    
    _, ablated_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_shuffled, loc, prop, menu))
    
    # Calculate CIC: divergence in deterministic policy
    true_actions = jnp.argmax(true_logits, axis=-1)
    ablated_actions = jnp.argmax(ablated_logits, axis=-1)
    
    cic = jnp.mean((true_actions != ablated_actions).astype(jnp.float32))
    
    return cic


@functools.partial(jax.jit, static_argnames=("doer_apply_fn",))
def compute_two_doer_cic(
    doer_apply_fn: Callable,
    doer_params: dict,
    transition_batch,
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    rng: jax.random.PRNGKey,
) -> jnp.ndarray:
    """
    Computes CIC for the two-Doer setting by ablating each private message stream
    and measuring how often the shared Doer policy changes its greedy action.
    """

    def scan_fn(carry, step_data):
        doer_carry = carry
        msg, loc, prop, menu = step_data
        batch_size, num_doers = loc.shape[:2]
        flat_loc = loc.reshape((batch_size * num_doers,) + loc.shape[2:])
        flat_prop = prop.reshape((batch_size * num_doers,) + prop.shape[2:])
        flat_msg = msg.reshape((batch_size * num_doers,) + msg.shape[2:])
        flat_menu = menu.reshape((batch_size * num_doers,) + menu.shape[2:])
        flat_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_carry, flat_logits = doer_apply_fn(
            {"params": doer_params},
            flat_carry,
            flat_loc,
            flat_prop,
            flat_msg,
            flat_menu,
        )
        next_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_carry,
        )
        logits = flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))
        return next_carry, logits

    loc = transition_batch.local_obs
    prop = transition_batch.proprioception
    msg_true = transition_batch.message
    menu = transition_batch.menu_images

    _, true_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_true, loc, prop, menu))

    flat_keys = jax.random.split(rng, msg_true.shape[1] * msg_true.shape[2])
    msg_shuffled = jnp.swapaxes(msg_true, 0, 1)
    msg_shuffled = jnp.swapaxes(msg_shuffled, 1, 2)
    msg_shuffled = msg_shuffled.reshape((-1, msg_true.shape[0], msg_true.shape[-1]))
    msg_shuffled = jax.vmap(
        lambda key, msgs: jax.random.permutation(key, msgs, axis=0)
    )(flat_keys, msg_shuffled)
    msg_shuffled = msg_shuffled.reshape(
        (msg_true.shape[1], msg_true.shape[2], msg_true.shape[0], msg_true.shape[-1])
    )
    msg_shuffled = jnp.swapaxes(msg_shuffled, 1, 2)
    msg_shuffled = jnp.swapaxes(msg_shuffled, 0, 1)

    _, ablated_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_shuffled, loc, prop, menu))

    true_actions = jnp.argmax(true_logits, axis=-1)
    ablated_actions = jnp.argmax(ablated_logits, axis=-1)
    cic = jnp.mean((true_actions != ablated_actions).astype(jnp.float32))
    return cic


In [ ]:
%%writefile train.py
import jax
import jax.numpy as jnp
import optax
from pathlib import Path
import flax.linen as nn
from flax.training.train_state import TrainState

# Import our custom modules
from models.seer import Seer
from models.doer import Doer
from envs.navix_wrapper import NavixGridWrapper, UNSET_POSITION
from envs.two_doer_grid import TwoDoerBottleneckEnv, UNSET_TWO_DOER_POSITIONS
from training.loop import (
    generate_trajectory_and_gae,
    generate_two_doer_trajectory_and_gae,
    make_rollout_step,
    make_two_doer_rollout_step,
)
from agents.mappo import (
    update_actor,
    update_actor_two_doer,
    update_critic,
    update_critic_two_doer,
    Transition,
)
import navix as nx
from eval.visualize import visualize_episode
from eval.metrics import compute_two_doer_cic
import numpy as np
import wandb
from PIL import Image


# For the sake of a complete script, here is a pragmatic, standard Critic
class GlobalCritic(nn.Module):
    """Evaluates the global state to guide learning (CTDE)."""
    output_dim: int = 2

    @nn.compact
    def __call__(self, global_map: jnp.ndarray) -> jnp.ndarray:
        x = nn.Conv(features=32, kernel_size=(3, 3))(global_map)
        x = nn.relu(x)
        x = x.reshape((x.shape[0], -1))
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        value = nn.Dense(features=self.output_dim)(x)
        return value


def reset_curriculum_batch(
    env,
    rng,
    num_envs,
    vision_radius,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    rng, env_rng = jax.random.split(rng)
    reset_keys = jax.random.split(env_rng, num_envs)
    env_obs, env_state = env.reset_batch(
        reset_keys,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )
    return rng, env_obs, env_state


def sample_curriculum_anchor(
    env,
    rng,
    vision_radius,
    control_mode,
    fixed_goal_position=UNSET_POSITION,
    exclude_start_position=UNSET_POSITION,
    max_attempts=512,
):
    for _ in range(max_attempts):
        rng, sample_rng = jax.random.split(rng)
        _, timestep = env.reset(
            sample_rng,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
        )
        start_position = env.player_position(timestep)
        if jnp.any(exclude_start_position < 0) or not bool(
            jnp.all(start_position == exclude_start_position)
        ):
            return rng, env.goal_position(timestep), start_position
    raise RuntimeError(
        "Failed to sample a new curriculum start position. "
        "This curriculum requires an environment with random starts, "
        "for example 'Navix-Empty-Random-8x8-v0'."
    )


def collect_message_action_trace(
    env,
    params,
    rng,
    seer,
    doer,
    vision_radius,
    max_steps,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    action_labels = ("turn_left", "turn_right", "forward")
    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(
        reset_rng,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)
    trace_lines = []
    done = False
    step_count = 0

    while not bool(done) and step_count < max_steps:
        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_view = obs["local_view"][None, ...]
        proprioception = obs["proprioception"][None, ...]

        target_images = obs["target_images"][None, ...]
        menu_images = obs["menu_images"][None, ...]

        seer_carry, message, _, _ = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
            target_images,
        )
        doer_carry, action_logits = doer.apply(
            {"params": params["doer"]},
            doer_carry,
            local_view,
            proprioception,
            message,
            menu_images[:, 0],
        )

        message_np = np.asarray(message[0]).round(3).tolist()
        action = int(jnp.argmax(action_logits[0]))
        action_label = action_labels[action] if action < len(action_labels) else str(action)
        player_position = np.asarray(env.player_position(state)).tolist()
        trace_lines.append(
            f"t={step_count:02d} pos={player_position} msg={message_np} action={action_label}"
        )

        rng, step_rng = jax.random.split(rng)
        obs, state, _, done, info = env.step(
            step_rng,
            state,
            jnp.asarray(action, dtype=jnp.int32),
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )
        step_count += 1

    final_status = "solved" if bool(done) and float(info["task_reward"]) > 0.0 else "stopped"
    trace_lines.append(f"end={final_status} steps={step_count}")
    return rng, trace_lines


def print_communication_trace(
    env,
    params,
    rng,
    seer,
    doer,
    vision_radius,
    max_steps,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
    label,
):
    rng, trace_rng = jax.random.split(rng)
    rng, trace_lines = collect_message_action_trace(
        env,
        params,
        trace_rng,
        seer,
        doer,
        vision_radius,
        max_steps,
        control_mode,
        fixed_goal_position,
        fixed_start_position,
    )
    print(f"Communication trace ({label}):")
    for line in trace_lines:
        print(line)
    return rng


def evaluate_greedy_episode(
    env,
    params,
    rng,
    seer,
    doer,
    vision_radius,
    max_steps,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(
        reset_rng,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)
    done = False
    step_count = 0
    solved = False

    while not bool(done) and step_count < max_steps:
        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_view = obs["local_view"][None, ...]
        proprioception = obs["proprioception"][None, ...]

        target_images = obs["target_images"][None, ...]
        menu_images = obs["menu_images"][None, ...]

        seer_carry, message, _, seer_nav_logits = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
            target_images,
        )

        if int(control_mode) == env.SEER_NAV_PHASE:
            action = jnp.argmax(seer_nav_logits[0]).astype(jnp.int32)
        else:
            doer_carry, action_logits = doer.apply(
                {"params": params["doer"]},
                doer_carry,
                local_view,
                proprioception,
                message,
                menu_images[:, 0],
            )
            action = jnp.argmax(action_logits[0]).astype(jnp.int32)

        rng, step_rng = jax.random.split(rng)
        obs, state, _, done, info = env.step(
            step_rng,
            state,
            action,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )
        solved = solved or bool(done) and float(info["task_reward"]) > 0.0
        step_count += 1

    return rng, solved


def initialize_two_doer_carry(doer, num_envs, num_doers, hidden_size):
    flat_carry = doer.initialize_carry(batch_size=num_envs * num_doers, hidden_size=hidden_size)
    return jax.tree_util.tree_map(
        lambda x: x.reshape((num_envs, num_doers, hidden_size)),
        flat_carry,
    )


def reset_two_doer_batch(env, rng, num_envs, fixed_positions):
    rng, env_rng = jax.random.split(rng)
    reset_keys = jax.random.split(env_rng, num_envs)
    env_obs, env_state = env.reset_batch(reset_keys, fixed_positions=fixed_positions)
    return rng, env_obs, env_state


def sample_two_doer_curriculum_anchor(
    env,
    rng,
    exclude_positions=UNSET_TWO_DOER_POSITIONS,
    max_attempts=512,
):
    for _ in range(max_attempts):
        rng, sample_rng = jax.random.split(rng)
        _, state = env.reset(sample_rng)
        if jnp.any(exclude_positions < 0) or not bool(jnp.all(state.positions == exclude_positions)):
            return rng, state.positions
    raise RuntimeError("Failed to sample a new two-doer curriculum start configuration.")


def _run_two_doer_greedy_episode(
    env,
    params,
    rng,
    seer,
    doer,
    max_steps,
    fixed_positions=UNSET_TWO_DOER_POSITIONS,
    capture_trace=False,
):
    action_labels = ("stay", "up", "right", "down", "left")
    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(reset_rng, fixed_positions=fixed_positions)

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = initialize_two_doer_carry(doer, num_envs=1, num_doers=env.num_doers, hidden_size=128)
    done = False
    success = False
    step_count = 0
    trace_lines = []

    while not bool(done) and step_count < max_steps:
        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_views = obs["local_views"][None, ...]
        proprioceptions = obs["proprioceptions"][None, ...]

        target_images = obs["target_images"][None, ...]
        menu_images = obs["menu_images"][None, ...]

        seer_carry, messages, _, _ = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
            target_images,
        )

        batch_size, num_doers = local_views.shape[:2]
        flat_local_views = local_views.reshape((batch_size * num_doers,) + local_views.shape[2:])
        flat_proprioceptions = proprioceptions.reshape(
            (batch_size * num_doers,) + proprioceptions.shape[2:]
        )
        flat_messages = messages.reshape((batch_size * num_doers,) + messages.shape[2:])
        flat_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_doer_carry, flat_logits = doer.apply(
            {"params": params["doer"]},
            flat_doer_carry,
            flat_local_views,
            flat_proprioceptions,
            flat_messages,
            menu_images.reshape((batch_size * num_doers,) + menu_images.shape[2:]),
        )
        doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_doer_carry,
        )
        actions = jnp.argmax(
            flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))[0],
            axis=-1,
        ).astype(jnp.int32)

        if capture_trace:
            positions = np.asarray(state.positions).tolist()
            message_np = np.asarray(messages[0]).round(3).tolist()
            action_names = [
                action_labels[int(action)] if int(action) < len(action_labels) else str(int(action))
                for action in np.asarray(actions).tolist()
            ]
            trace_lines.append(
                " ".join(
                    [
                        f"t={step_count:02d}",
                        f"pos_a={positions[0]}",
                        f"pos_b={positions[1]}",
                        f"msg_a={message_np[0]}",
                        f"act_a={action_names[0]}",
                        f"msg_b={message_np[1]}",
                        f"act_b={action_names[1]}",
                    ]
                )
            )

        rng, step_rng = jax.random.split(rng)
        obs, state, _, done, info = env.step(
            step_rng,
            state,
            actions,
            fixed_positions=fixed_positions,
        )
        success = success or bool(info["success"])
        step_count += 1

    if capture_trace:
        final_status = "solved" if bool(info["success"]) else "stopped"
        trace_lines.append(f"end={final_status} steps={step_count}")
    return rng, success, trace_lines


def evaluate_two_doer_greedy_episode(
    env,
    params,
    rng,
    seer,
    doer,
    max_steps,
    fixed_positions=UNSET_TWO_DOER_POSITIONS,
):
    rng, success, _ = _run_two_doer_greedy_episode(
        env,
        params,
        rng,
        seer,
        doer,
        max_steps,
        fixed_positions=fixed_positions,
        capture_trace=False,
    )
    return rng, success


def collect_two_doer_message_action_trace(
    env,
    params,
    rng,
    seer,
    doer,
    max_steps,
    fixed_positions=UNSET_TWO_DOER_POSITIONS,
):
    rng, _, trace_lines = _run_two_doer_greedy_episode(
        env,
        params,
        rng,
        seer,
        doer,
        max_steps,
        fixed_positions=fixed_positions,
        capture_trace=True,
    )
    return rng, trace_lines


def print_two_doer_communication_trace(
    env,
    params,
    rng,
    seer,
    doer,
    max_steps,
    label,
    fixed_positions=UNSET_TWO_DOER_POSITIONS,
):
    rng, trace_rng = jax.random.split(rng)
    rng, trace_lines = collect_two_doer_message_action_trace(
        env,
        params,
        trace_rng,
        seer,
        doer,
        max_steps,
        fixed_positions,
    )
    print(f"Two-doer communication trace ({label}):")
    for line in trace_lines:
        print(line)
    return rng


def flatten_message_codes(message_batch, fsq_levels):
    levels = np.asarray(fsq_levels, dtype=np.int32)
    multipliers = np.ones_like(levels)
    for idx in range(len(levels) - 2, -1, -1):
        multipliers[idx] = multipliers[idx + 1] * levels[idx + 1]

    messages = np.rint(np.asarray(message_batch)).astype(np.int32)
    messages = np.clip(messages, 0, levels - 1)
    return (messages * multipliers).sum(axis=-1).astype(np.int32).reshape(-1)


def compute_message_stats(message_batch, fsq_levels):
    message_codes = flatten_message_codes(message_batch, fsq_levels)
    num_codes = int(np.prod(np.asarray(fsq_levels, dtype=np.int32)))
    counts = np.bincount(message_codes, minlength=num_codes).astype(np.float32)
    total = max(int(counts.sum()), 1)
    probs = counts / float(total)
    nonzero_probs = probs[probs > 0.0]
    entropy = float(-(nonzero_probs * np.log(nonzero_probs)).sum()) if nonzero_probs.size else 0.0
    max_entropy = float(np.log(num_codes)) if num_codes > 1 else 0.0
    normalized_entropy = entropy / max_entropy if max_entropy > 0.0 else 0.0
    unique_codes = int((counts > 0).sum())
    return {
        "message_codes": message_codes,
        "message_code_probs": probs,
        "rollout_message_entropy": entropy,
        "rollout_message_entropy_normalized": normalized_entropy,
        "rollout_message_unique_codes": unique_codes,
        "rollout_message_num_codes": num_codes,
    }


def log_curriculum_visualization(
    env,
    params,
    rng,
    seer,
    doer,
    config,
    update,
    phase_label,
    vision_radius,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    viz_path = Path(config["visualize_dir"]) / f"{phase_label}_{update:05d}.gif"
    output_path, solved = visualize_episode(
        env,
        params,
        rng,
        seer,
        doer,
        filename=str(viz_path),
        vision_radius=vision_radius,
        max_steps=config["visualize_max_steps"],
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )
    wandb.log(
        {
            "curriculum_reset_episode": wandb.Video(str(output_path), format="gif"),
            "curriculum_reset_episode_solved": int(solved),
        },
        commit=False,
    )


def save_two_doer_initial_visualization(env, state, config):
    output_path = Path(config["visualize_dir"]) / "two_doer_initial_layout.png"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame = env.render(state)
    Image.fromarray(frame).save(output_path)
    wandb.log(
        {
            "two_doer_initial_layout": wandb.Image(str(output_path)),
            "two_doer_initial_layout_step": 0,
        },
        commit=True,
    )
    print(f"Initial two-doer layout saved to {output_path}")
    return output_path


def print_two_doer_start_positions_banner(fixed_positions, label="TWO-DOER STARTS"):
    positions_np = np.asarray(fixed_positions).tolist()
    print("")
    print("=" * 72)
    print(label)
    print(
        f"Doer A: ({positions_np[0][0]}, {positions_np[0][1]}) | "
        f"Doer B: ({positions_np[1][0]}, {positions_np[1][1]})"
    )
    print("=" * 72)
    print("")


def print_two_doer_perception_level_banner(level, label="TWO-DOER PERCEPTION LEVEL"):
    print("")
    print("=" * 72)
    print(label)
    print(f"doer_perception_level={level}")
    print("=" * 72)
    print("")


def visualize_two_doer_episode(
    env,
    params,
    rng,
    seer,
    doer,
    config,
    update,
    fixed_positions=UNSET_TWO_DOER_POSITIONS,
):
    output_path = Path(config["visualize_dir"]) / f"two_doer_eval_{update:05d}.gif"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(reset_rng, fixed_positions=fixed_positions)
    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = initialize_two_doer_carry(
        doer,
        num_envs=1,
        num_doers=env.num_doers,
        hidden_size=128,
    )
    done = False
    step_count = 0
    success = False
    frames = []

    while not bool(done) and step_count < config["episode_max_steps"]:
        frame = env.render(state)
        frames.append(Image.fromarray(frame))

        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_views = obs["local_views"][None, ...]
        proprioceptions = obs["proprioceptions"][None, ...]

        target_images = obs["target_images"][None, ...]
        menu_images = obs["menu_images"][None, ...]

        seer_carry, messages, _, _ = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
            target_images,
        )

        batch_size, num_doers = local_views.shape[:2]
        flat_local_views = local_views.reshape((batch_size * num_doers,) + local_views.shape[2:])
        flat_proprioceptions = proprioceptions.reshape(
            (batch_size * num_doers,) + proprioceptions.shape[2:]
        )
        flat_messages = messages.reshape((batch_size * num_doers,) + messages.shape[2:])
        flat_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size * num_doers,) + x.shape[2:]),
            doer_carry,
        )
        next_flat_doer_carry, flat_logits = doer.apply(
            {"params": params["doer"]},
            flat_doer_carry,
            flat_local_views,
            flat_proprioceptions,
            flat_messages,
            menu_images.reshape((batch_size * num_doers,) + menu_images.shape[2:]),
        )
        doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((batch_size, num_doers) + x.shape[1:]),
            next_flat_doer_carry,
        )
        actions = jnp.argmax(
            flat_logits.reshape((batch_size, num_doers, flat_logits.shape[-1]))[0],
            axis=-1,
        ).astype(jnp.int32)

        rng, step_rng = jax.random.split(rng)
        obs, state, _, done, info = env.step(
            step_rng,
            state,
            actions,
            fixed_positions=fixed_positions,
        )
        success = success or bool(info["success"])
        step_count += 1

    if frames:
        frames[0].save(
            output_path,
            save_all=True,
            append_images=frames[1:],
            duration=150,
            loop=0,
        )
        wandb.log(
            {
                "two_doer_eval_episode": wandb.Video(str(output_path), format="gif"),
                "two_doer_eval_episode_solved": int(success),
                "two_doer_eval_episode_step": int(update),
            },
            commit=True,
        )
        print(f"Two-doer eval episode saved to {output_path}")

    return rng, output_path, success


def run_two_doer_training(config):
    rng = jax.random.PRNGKey(config["seed"])
    rng, seer_init_rng, doer_init_rng, critic_init_rng, reset_rng = jax.random.split(rng, 5)

    env = TwoDoerBottleneckEnv(
        grid_height=config["grid_height"],
        grid_width=config["grid_width"],
        local_view_size=config["local_view_size"],
        corridor_length=config["corridor_length"],
        max_steps=config["episode_max_steps"],
        progress_reward_scale=config["progress_reward_scale"],
        goal_reward=config["goal_reward"],
        step_penalty=config["step_penalty"],
        wall_penalty=config["wall_penalty"],
        collision_penalty=config["collision_penalty"],
        doer_perception_level=config["doer_perception_level"],
    )
    curriculum_active = (
        config["use_two_doer_start_curriculum"]
        and not config["two_doer_random_starts_only"]
    )
    current_start_success_streak = 0
    mastered_start_positions = 0
    fixed_positions = UNSET_TWO_DOER_POSITIONS
    if curriculum_active:
        rng, fixed_positions = sample_two_doer_curriculum_anchor(env, rng)
        print_two_doer_start_positions_banner(fixed_positions, label="INITIAL TWO-DOER STARTS")
    print_two_doer_perception_level_banner(
        env.doer_perception_level,
        label="INITIAL TWO-DOER PERCEPTION LEVEL",
    )

    rng, env_obs, env_state = reset_two_doer_batch(
        env,
        rng,
        config["num_envs"],
        fixed_positions,
    )
    save_two_doer_initial_visualization(env, jax.tree_util.tree_map(lambda x: x[0], env_state), config)

    seer = Seer(
        fsq_levels=config["fsq_levels"],
        num_actions=env.num_actions,
        num_message_heads=env.num_doers,
    )
    doer = Doer(fsq_levels=config["fsq_levels"], num_actions=env.num_actions)
    critic = GlobalCritic(output_dim=1)

    dummy_map = env_obs["global_map"][:1]
    dummy_sym = env_obs["symbolic_state"][:1]
    dummy_local = env_obs["local_views"][:1, 0]
    dummy_prop = env_obs["proprioceptions"][:1, 0]
    dummy_msg = jnp.zeros((1, len(config["fsq_levels"])))
    init_seer_carry = seer.initialize_carry(1, 128)
    init_doer_carry = doer.initialize_carry(1, 128)

    dummy_target = env_obs["target_images"][:1]
    dummy_menu = env_obs["menu_images"][:1, 0]
    seer_params = seer.init(seer_init_rng, init_seer_carry, dummy_map, dummy_sym, dummy_target)["params"]
    doer_params = doer.init(doer_init_rng, init_doer_carry, dummy_local, dummy_prop, dummy_msg, dummy_menu)["params"]
    critic_params = critic.init(critic_init_rng, dummy_map)["params"]

    seer_carry = seer.initialize_carry(config["num_envs"], 128)
    doer_carry = initialize_two_doer_carry(
        doer,
        num_envs=config["num_envs"],
        num_doers=env.num_doers,
        hidden_size=128,
    )
    params = {"seer": seer_params, "doer": doer_params, "critic": critic_params}

    tx = optax.chain(
        optax.clip_by_global_norm(0.5),
        optax.adam(learning_rate=config["learning_rate"], eps=1e-5),
    )
    actor_state = TrainState.create(
        apply_fn=None,
        params={"seer": seer_params, "doer": doer_params},
        tx=tx,
    )
    critic_state = TrainState.create(
        apply_fn=critic.apply,
        params=critic_params,
        tx=tx,
    )
    step_fn = make_two_doer_rollout_step(
        env,
        seer.apply,
        doer.apply,
        critic.apply,
    )
    num_updates = config["total_timesteps"] // (config["num_steps"] * config["num_envs"])

    print(
        "Starting training... "
        f"(task=two_doer_bottleneck, num_doers={env.num_doers}, fsq_levels={config['fsq_levels']}, "
        f"curriculum={'fixed_starts' if curriculum_active else 'random_starts'})"
    )

    for update in range(num_updates):
        rng, rollout_rng = jax.random.split(rng)
        init_seer_carry = seer_carry
        init_doer_carry = doer_carry

        final_runner_state, trajectory_batch = generate_two_doer_trajectory_and_gae(
            params,
            rollout_rng,
            env_obs,
            env_state,
            seer_carry,
            doer_carry,
            fixed_positions,
            config["num_steps"],
            step_fn,
            critic.apply,
        )
        params, seer_carry, doer_carry, env_state, env_obs, _, _ = final_runner_state

        rng, actor_rng, critic_rng = jax.random.split(rng, 3)
        batched_trajectory = jax.tree_util.tree_map(
            lambda x: jnp.swapaxes(x, 0, 1),
            trajectory_batch,
        )
        actor_state, actor_metrics = update_actor_two_doer(
            actor_state,
            batched_trajectory,
            init_seer_carry,
            init_doer_carry,
            seer.apply,
            doer.apply,
            actor_rng,
            tuple(config["fsq_levels"]),
            jnp.asarray(config["seer_entropy_coef"], dtype=jnp.float32),
        )
        critic_state, critic_metrics = update_critic_two_doer(
            critic_state,
            batched_trajectory,
            critic.apply,
            critic_rng,
        )
        params["seer"] = actor_state.params["seer"]
        params["doer"] = actor_state.params["doer"]
        params["critic"] = critic_state.params

        success_events = jnp.logical_and(
            trajectory_batch.done,
            trajectory_batch.task_reward > 0.0,
        )
        completed_episodes = trajectory_batch.done.astype(jnp.int32).sum()
        rollout_num_successes = success_events.astype(jnp.int32).sum()
        rollout_success_rate = jnp.where(
            completed_episodes > 0,
            rollout_num_successes.astype(jnp.float32)
            / completed_episodes.astype(jnp.float32),
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        message_stats = compute_message_stats(trajectory_batch.message, config["fsq_levels"])
        rng, cic_rng = jax.random.split(rng)
        cic_score = compute_two_doer_cic(
            doer.apply,
            params["doer"],
            trajectory_batch,
            init_doer_carry,
            cic_rng,
        )

        if update % 10 == 0:
            wandb.log(
                {
                    "task_variant": config["task_variant"],
                    "doer_perception_level": env.doer_perception_level,
                    "curriculum_fixed_starts": int(curriculum_active),
                    "curriculum_eval_gate_passed": int(
                        rollout_success_rate >= config["curriculum_rollout_success_threshold"]
                    ),
                    "mastered_start_positions": mastered_start_positions,
                    "rollout_success_rate": rollout_success_rate,
                    "team_reward": trajectory_batch.reward.mean(),
                    "task_reward": trajectory_batch.task_reward.mean(),
                    "progress_reward_doer_a": trajectory_batch.progress_reward_per_doer[..., 0].mean(),
                    "progress_reward_doer_b": trajectory_batch.progress_reward_per_doer[..., 1].mean(),
                    "step_penalty": trajectory_batch.step_penalty_component.mean(),
                    "wall_penalty": trajectory_batch.wall_penalty_component.mean(),
                    "collision_penalty": trajectory_batch.collision_penalty_component.mean(),
                    "cic_score": cic_score,
                    "rollout_message_entropy_normalized": message_stats["rollout_message_entropy_normalized"],
                    "rollout_message_unique_codes": message_stats["rollout_message_unique_codes"],
                    "critic_loss": critic_metrics.get("critic_loss", 0.0),
                    "seer_grad_norm": actor_metrics.get("seer_grad_norm", 0.0),
                    "doer_grad_norm": actor_metrics.get("doer_grad_norm", 0.0),
                    "message_distribution": wandb.Histogram(
                        np.asarray(message_stats["message_codes"])
                    ),
                }
            )
            print(
                f"Update {update}/{num_updates} | "
                f"Curriculum: {'fixed' if curriculum_active else 'random'} | "
                f"EvalGate: {int(rollout_success_rate >= config['curriculum_rollout_success_threshold'])} | "
                f"Mastered: {mastered_start_positions}/"
                f"{config['two_doer_required_start_positions']} | "
                f"SuccessRate: {float(rollout_success_rate):.3f} | "
                f"TeamReward: {trajectory_batch.reward.mean():.3f} | "
                f"Task: {trajectory_batch.task_reward.mean():.3f} | "
                f"ProgA: {trajectory_batch.progress_reward_per_doer[..., 0].mean():.3f} | "
                f"ProgB: {trajectory_batch.progress_reward_per_doer[..., 1].mean():.3f} | "
                f"Step: {trajectory_batch.step_penalty_component.mean():.3f} | "
                f"Wall: {trajectory_batch.wall_penalty_component.mean():.3f} | "
                f"Coll: {trajectory_batch.collision_penalty_component.mean():.3f} | "
                f"MsgH: {message_stats['rollout_message_entropy_normalized']:.3f} | "
                f"MsgUsed: {message_stats['rollout_message_unique_codes']}/"
                f"{message_stats['rollout_message_num_codes']} | "
                f"CIC: {float(cic_score):.3f} | "
                f"SeerGrad: {actor_metrics.get('seer_grad_norm', 0.0):.4f} | "
                f"DoerGrad: {actor_metrics.get('doer_grad_norm', 0.0):.4f}"
            )

        if update > 0 and update % config["eval_every"] == 0:
            gate_passed = bool(
                float(rollout_success_rate) >= config["curriculum_rollout_success_threshold"]
            )
            greedy_solved = False
            if gate_passed:
                rng, greedy_solved, trace_lines = _run_two_doer_greedy_episode(
                    env,
                    params,
                    rng,
                    seer,
                    doer,
                    config["episode_max_steps"],
                    fixed_positions=fixed_positions,
                    capture_trace=True,
                )
                print(
                    f"Greedy eval @ {update}: "
                    f"gate_passed=1 solved={int(greedy_solved)}"
                )
                print(f"Two-doer communication trace (update_{update}):")
                for line in trace_lines:
                    print(line)
            else:
                print(
                    f"Greedy eval @ {update}: "
                    f"gate_passed=0 skipped "
                    f"(rollout_success_rate={float(rollout_success_rate):.3f})"
                )
            wandb.log(
                {
                    "curriculum_eval_gate_passed": int(gate_passed),
                    "greedy_episode_solved": int(greedy_solved),
                }
            )

            if curriculum_active:
                if greedy_solved:
                    mastered_start_positions += 1
                    previous_fixed_positions = fixed_positions

                    if mastered_start_positions >= config["two_doer_required_start_positions"]:
                        curriculum_active = False
                        fixed_positions = UNSET_TWO_DOER_POSITIONS
                        if env.doer_perception_level < config["max_doer_perception_level"]:
                            env.doer_perception_level = config["max_doer_perception_level"]
                            config["doer_perception_level"] = env.doer_perception_level
                            print_two_doer_perception_level_banner(
                                env.doer_perception_level,
                                label="NEW TWO-DOER PERCEPTION LEVEL",
                            )
                        print("")
                        print("=" * 72)
                        print("TWO-DOER CURRICULUM COMPLETE: random starts enabled")
                        print("=" * 72)
                        print("")
                    else:
                        rng, fixed_positions = sample_two_doer_curriculum_anchor(
                            env,
                            rng,
                            exclude_positions=previous_fixed_positions,
                        )
                        print_two_doer_start_positions_banner(
                            fixed_positions,
                            label="NEW TWO-DOER STARTS",
                        )

                    rng, env_obs, env_state = reset_two_doer_batch(
                        env,
                        rng,
                        config["num_envs"],
                        fixed_positions,
                    )
                    seer_carry = seer.initialize_carry(config["num_envs"], 128)
                    doer_carry = initialize_two_doer_carry(
                        doer,
                        num_envs=config["num_envs"],
                        num_doers=env.num_doers,
                        hidden_size=128,
                    )
                else:
                    current_start_success_streak = 0

        if update > 0 and update % config["visualize_every"] == 0:
            rng, _, _ = visualize_two_doer_episode(
                env,
                params,
                rng,
                seer,
                doer,
                config,
                update,
                fixed_positions=fixed_positions,
            )


def main():
    # 1. Configuration and Logging
    config = {
        "task_variant": "two_doer_bottleneck",
        "learning_rate": 3e-4,
        "num_envs": 16,
        "num_steps": 64,
        "total_timesteps": 3_000_000,
        "env_id": "Navix-Empty-Random-8x8-v0",
        "fsq_levels": [2, 2, 2, 2],
        "seed": 42,
        "grid_height": 10,
        "grid_width": 12,
        "local_view_size": 3,
        "corridor_length": 3,
        "episode_max_steps": 48,
        "goal_reward": 1.0,
        "follow_reward_scale": 0.1,
        "progress_reward_scale": 0.1,
        "cic_coef": 0.01,
        "seer_entropy_coef": 0.05,
        "doer_perception_level": 2,
        "max_doer_perception_level": 3,
        "curriculum_success_streak": 3,
        "curriculum_eval_every": 25,
        "eval_every": 300,
        "curriculum_rollout_success_threshold": 0.97,
        "visualize_every": 200,
        "use_two_doer_start_curriculum": True,
        "two_doer_random_starts_only": False,
        "two_doer_required_start_positions": 3,
        "use_seer_nav_phase": False,
        "seer_required_start_positions": 5,
        "communication_start_positions_per_level": 5,
        "release_goal_after_max_level": True,
        "min_start_distance": 1.0,
        "step_penalty": 0.03,
        "bump_penalty": 0.1,
        "wall_penalty": 0.02,
        "collision_penalty": 0.05,
        "visualize_max_steps": 30,
        "visualize_dir": "artifacts/episodes",
    }
    
    wandb.init(entity="eleftheriaklk-ucl", project="brian_test", config=config)

    print(f"backend: {jax.default_backend()}")
    print(f"devices: {jax.devices()}")

    if config["task_variant"] == "two_doer_bottleneck":
        run_two_doer_training(config)
        return
    
    # 2. PRNG Key Initialization
    # JAX requires explicit, rigorous management of randomness
    rng = jax.random.PRNGKey(config["seed"])
    rng, seer_init_rng, doer_init_rng, critic_init_rng, env_rng = jax.random.split(rng, 5)

    # 3. Environment Instantiation
    raw_env = nx.make(config["env_id"])
    env = NavixGridWrapper(
        raw_env,
        progress_reward_scale=config["progress_reward_scale"],
        min_start_distance=config["min_start_distance"],
        step_penalty=config["step_penalty"],
        bump_penalty=config["bump_penalty"],
        doer_perception_level=config["doer_perception_level"],
    )
    seer_nav_mode = jnp.array(env.SEER_NAV_PHASE, dtype=jnp.int32)
    communication_mode = jnp.array(env.COMMUNICATION_PHASE, dtype=jnp.int32)
    control_mode = (
        seer_nav_mode
        if config["use_seer_nav_phase"]
        else communication_mode
    )

    # 4. Initial Environment Reset
    fixed_goal_position = UNSET_POSITION
    fixed_start_position = UNSET_POSITION
    rng, fixed_goal_position, fixed_start_position = sample_curriculum_anchor(
        env,
        rng,
        vision_radius=jnp.array(3.0),
        control_mode=control_mode,
    )
    env.doer_perception_level = config["doer_perception_level"]
    rng, env_obs, env_state = reset_curriculum_batch(
        env,
        rng,
        config["num_envs"],
        vision_radius=jnp.array(3.0),
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    # 5. Network Instantiation
    seer = Seer(fsq_levels=config["fsq_levels"])
    doer = Doer(fsq_levels=config["fsq_levels"], num_actions=env.num_actions)
    critic = GlobalCritic()

    # 6. Parameter Initialization (Dummy Forward Passes)
    # We must pass data of the correct shape to initialize the Flax parameters
    dummy_map = env_obs["global_map"][:1]
    dummy_sym = env_obs["symbolic_state"][:1]
    dummy_local = env_obs["local_view"][:1]
    dummy_prop = env_obs["proprioception"][:1]
    dummy_msg = jnp.zeros((1, len(config["fsq_levels"])))
    
    init_seer_carry = seer.initialize_carry(1, 128)
    init_doer_carry = doer.initialize_carry(1, 128)

    dummy_target = env_obs["target_images"][:1]
    dummy_menu = env_obs["menu_images"][:1, 0]
    seer_params = seer.init(seer_init_rng, init_seer_carry, dummy_map, dummy_sym, dummy_target)["params"]
    doer_params = doer.init(doer_init_rng, init_doer_carry, dummy_local, dummy_prop, dummy_msg, dummy_menu)["params"]
    critic_params = critic.init(critic_init_rng, dummy_map)["params"]

    seer_carry = seer.initialize_carry(config["num_envs"], 128)
    doer_carry = doer.initialize_carry(config["num_envs"], 128)

    # Group parameters for the execution loop
    params = {"seer": seer_params, "doer": doer_params, "critic": critic_params}

    # 6. Optimizer and TrainState Setup
    # Optax provides the gradient transformation tools
    tx = optax.chain(
        optax.clip_by_global_norm(0.5),
        optax.adam(learning_rate=config["learning_rate"], eps=1e-5)
    )

    # Since the Seer and Doer act cooperatively to generate the trajectory,
    # we can conceptually treat them as a single "Actor" policy for the optimizer.
    # In a more advanced setup, you might give them separate optimizers.
    actor_state = TrainState.create(
        apply_fn=None, # We use specific apply fns in the update step
        params={"seer": seer_params, "doer": doer_params},
        tx=tx
    )
    
    critic_state = TrainState.create(
        apply_fn=critic.apply,
        params=critic_params,
        tx=tx
    )

    step_fn = make_rollout_step(
        env,
        seer.apply,
        doer.apply,
        critic.apply,
        follow_reward_scale=config["follow_reward_scale"],
    )
    current_start_success_streak = 0
    seer_mastered_starts = 0
    communication_mastered_starts = 0
    goal_randomization_enabled = False

    # 8. The Main Training Loop
    num_updates = config["total_timesteps"] // (config["num_steps"] * config["num_envs"])
    
    print(
        "Starting training... "
        f"(phase={'seer_nav' if config['use_seer_nav_phase'] else 'communication'}, "
        f"doer_perception_level={config['doer_perception_level']})"
    )
    for update in range(num_updates):
        rng, rollout_rng = jax.random.split(rng)
        
        # Curriculums
        vision_radius = jnp.clip(3.0 - 2.0 * (update / 1000.0), 1.0, 3.0)
        seer_entropy_coef = jnp.clip(0.1 - 0.09 * (update / 1000.0), 0.01, 0.1)
        
        # A. Collect Trajectory
        init_seer_carry = seer_carry
        init_doer_carry = doer_carry
        
        final_runner_state, trajectory_batch = generate_trajectory_and_gae(
            params,
            rollout_rng,
            env_obs,
            env_state,
            seer_carry,
            doer_carry,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
            jnp.array(config["cic_coef"], dtype=jnp.float32),
            config["num_steps"],
            step_fn, critic.apply, doer.apply
        )
        
        # Extract for next loop iteration
        params, seer_carry, doer_carry, env_state, env_obs, _, _, _, _, _ = final_runner_state
        
        # B. Generalized Advantage Estimation (GAE)
        # GAE is now calculated inside generate_trajectory_and_gae
        
        # C. Update Networks
        rng, actor_rng, critic_rng = jax.random.split(rng, 3)
        
        batched_trajectory = jax.tree_util.tree_map(
            lambda x: jnp.swapaxes(x, 0, 1),
            trajectory_batch
        )
        batched_seer_carry = init_seer_carry
        batched_doer_carry = init_doer_carry

        actor_state, actor_metrics = update_actor(
            actor_state, batched_trajectory, batched_seer_carry, batched_doer_carry, 
            seer.apply,
            doer.apply,
            actor_rng,
            control_mode,
            tuple(config["fsq_levels"]),
            seer_entropy_coef,
        )
        critic_state, critic_metrics = update_critic(
            critic_state, batched_trajectory, critic.apply, critic_rng
        )
        
        # Sync updated parameters back to the params dictionary for the next rollout
        params["seer"] = actor_state.params["seer"]
        params["doer"] = actor_state.params["doer"]
        params["critic"] = critic_state.params

        success_events = jnp.logical_and(
            trajectory_batch.done,
            trajectory_batch.task_reward > 0.0,
        )
        completed_episodes = trajectory_batch.done.astype(jnp.int32).sum()
        rollout_num_successes = success_events.astype(jnp.int32).sum()
        rollout_success_rate = jnp.where(
            completed_episodes > 0,
            rollout_num_successes.astype(jnp.float32)
            / completed_episodes.astype(jnp.float32),
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        message_stats = compute_message_stats(trajectory_batch.message, config["fsq_levels"])
        cic_score = float(trajectory_batch.cic_score.mean())
        
        # D. Logging
        if update % 10 == 0:
            if int(control_mode) == env.SEER_NAV_PHASE:
                phase_label = "seer_nav"
            elif goal_randomization_enabled:
                phase_label = "communication_random_full"
            else:
                phase_label = "communication_random_start"
            phase_indicators = {
                "phase_seer_nav": int(phase_label == "seer_nav"),
                "phase_communication_random_start": int(
                    phase_label == "communication_random_start"
                ),
                "phase_communication_random_full": int(
                    phase_label == "communication_random_full"
                ),
            }
            wandb.log({
                "phase_label": phase_label,
                "doer_perception_level": config["doer_perception_level"],
                "current_start_success_streak": current_start_success_streak,
                "rollout_success_rate": rollout_success_rate,
                "task_reward": trajectory_batch.task_reward.mean(),
                "progress_reward": trajectory_batch.progress_reward.mean(),
                "cic_score": cic_score,
                "rollout_message_entropy_normalized": message_stats["rollout_message_entropy_normalized"],
                "rollout_message_unique_codes": message_stats["rollout_message_unique_codes"],
                "critic_loss": critic_metrics.get("critic_loss", 0.0),
                "message_distribution": wandb.Histogram(
                    np.asarray(message_stats["message_codes"])
                ),
                **phase_indicators,
            })
            print(
                f"Update {update}/{num_updates} | "
                f"Phase: {phase_label} | "
                f"Level: {config['doer_perception_level']} | "
                f"Start: ({int(fixed_start_position[0])}, {int(fixed_start_position[1])}) | "
                f"Goal: ({int(fixed_goal_position[0])}, {int(fixed_goal_position[1])}) | "
                f"Streak: {current_start_success_streak} | "
                f"SuccessRate: {float(rollout_success_rate):.3f} | "
                f"Seer Reward: {trajectory_batch.reward[..., 0].mean():.3f} | "
                f"Doer Reward: {trajectory_batch.reward[..., 1].mean():.3f} | "
                f"Task: {trajectory_batch.task_reward.mean():.3f} | "
                f"Progress: {trajectory_batch.progress_reward.mean():.3f} | "
                f"Follow: {trajectory_batch.follow_reward.mean():.3f} | "
                f"MsgH: {message_stats['rollout_message_entropy_normalized']:.3f} | "
                f"MsgUsed: {message_stats['rollout_message_unique_codes']}/"
                f"{message_stats['rollout_message_num_codes']} | "
                f"Step: {trajectory_batch.step_penalty_component.mean():.3f} | "
                f"Bump: {trajectory_batch.bump_penalty_component.mean():.3f} | "
                f"Seer Grad: {actor_metrics.get('seer_grad_norm', 0.0):.4f} | "
                f"Doer Grad: {actor_metrics.get('doer_grad_norm', 0.0):.4f} | "
                f"CIC: {cic_score:.3f}"
            )
            
        if update > 0 and update % config["curriculum_eval_every"] == 0:
            rng, greedy_solved = evaluate_greedy_episode(
                env,
                params,
                rng,
                seer,
                doer,
                vision_radius,
                config["visualize_max_steps"],
                control_mode,
                fixed_goal_position,
                fixed_start_position,
            )
            current_start_success_streak = current_start_success_streak + 1 if greedy_solved else 0

            if int(control_mode) == env.SEER_NAV_PHASE:
                if current_start_success_streak >= config["curriculum_success_streak"]:
                    current_start_success_streak = 0
                    seer_mastered_starts += 1

                    if seer_mastered_starts >= config["seer_required_start_positions"]:
                        control_mode = communication_mode
                        env.doer_perception_level = config["doer_perception_level"]
                        communication_mastered_starts = 0
                        print("")
                        print("=" * 72)
                        print("NEW PHASE: communication")
                        print("=" * 72)
                        print("")
                        print("Seer navigation mastered on five starts; switching to communication phase.")
                    else:
                        print(
                            f"Seer mastered start {seer_mastered_starts}/"
                            f"{config['seer_required_start_positions']}."
                        )

                    rng, _, fixed_start_position = sample_curriculum_anchor(
                        env,
                        rng,
                        vision_radius,
                        control_mode,
                        fixed_goal_position=fixed_goal_position,
                        exclude_start_position=fixed_start_position,
                    )
                    rng, env_obs, env_state = reset_curriculum_batch(
                        env,
                        rng,
                        config["num_envs"],
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )
                    seer_carry = seer.initialize_carry(config["num_envs"], 128)
                    doer_carry = doer.initialize_carry(config["num_envs"], 128)
                    print("")
                    print("=" * 72)
                    print(f"NEW RANDOM POSITION: {tuple(np.asarray(fixed_start_position).tolist())}")
                    print("=" * 72)
                    print("")
                    rng, viz_rng = jax.random.split(rng)
                    log_curriculum_visualization(
                        env,
                        params,
                        viz_rng,
                        seer,
                        doer,
                        config,
                        update,
                        "seer_nav_reset",
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )

            elif int(control_mode) == env.COMMUNICATION_PHASE:
                if current_start_success_streak >= config["curriculum_success_streak"]:
                    current_start_success_streak = 0
                    communication_mastered_starts += 1

                    if (
                        communication_mastered_starts
                        >= config["communication_start_positions_per_level"]
                    ):
                        communication_mastered_starts = 0
                        mastered_sublevel = config["doer_perception_level"]
                        rng = print_communication_trace(
                            env,
                            params,
                            rng,
                            seer,
                            doer,
                            vision_radius,
                            config["visualize_max_steps"],
                            control_mode,
                            fixed_goal_position,
                            fixed_start_position,
                            f"mastered_level_{mastered_sublevel}",
                        )
                        if config["doer_perception_level"] < config["max_doer_perception_level"]:
                            config["doer_perception_level"] += 1
                            env.doer_perception_level = config["doer_perception_level"]
                            print("")
                            print("=" * 72)
                            print(
                                "NEW DOER PERCEPTION LEVEL: "
                                f"{config['doer_perception_level']}"
                            )
                            print("=" * 72)
                            print("")
                            print(
                                f"Curriculum advanced to doer_perception_level="
                                f"{config['doer_perception_level']}"
                            )
                        else:
                            if (
                                config["release_goal_after_max_level"]
                                and not goal_randomization_enabled
                            ):
                                goal_randomization_enabled = True
                                fixed_goal_position = UNSET_POSITION
                                fixed_start_position = UNSET_POSITION
                                print("")
                                print("=" * 72)
                                print("NEW SUBCASE: communication_random_full")
                                print("=" * 72)
                                print("")
                                print(
                                    "Max doer perception level mastered; releasing fixed goal "
                                    "and continuing in fully random Empty-Random-8x8."
                                )
                            else:
                                print("Max doer perception level reached; continuing with new starts.")

                    if goal_randomization_enabled:
                        previous_goal_position = fixed_goal_position
                        rng, fixed_goal_position, fixed_start_position = sample_curriculum_anchor(
                            env,
                            rng,
                            vision_radius,
                            control_mode,
                        )
                        if not np.array_equal(
                            np.asarray(previous_goal_position),
                            np.asarray(fixed_goal_position),
                        ):
                            print("")
                            print("=" * 72)
                            print(f"NEW RANDOM GOAL: {tuple(np.asarray(fixed_goal_position).tolist())}")
                            print("=" * 72)
                            print("")
                    else:
                        rng, _, fixed_start_position = sample_curriculum_anchor(
                            env,
                            rng,
                            vision_radius,
                            control_mode,
                            fixed_goal_position=fixed_goal_position,
                            exclude_start_position=fixed_start_position,
                        )
                    rng, env_obs, env_state = reset_curriculum_batch(
                        env,
                        rng,
                        config["num_envs"],
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )
                    seer_carry = seer.initialize_carry(config["num_envs"], 128)
                    doer_carry = doer.initialize_carry(config["num_envs"], 128)
                    print("")
                    print("=" * 72)
                    print(f"NEW RANDOM POSITION: {tuple(np.asarray(fixed_start_position).tolist())}")
                    print("=" * 72)
                    print("")
                    rng, viz_rng = jax.random.split(rng)
                    phase_label = (
                        "communication_random_full_reset"
                        if goal_randomization_enabled
                        else "communication_random_start_reset"
                    )
                    log_curriculum_visualization(
                        env,
                        params,
                        viz_rng,
                        seer,
                        doer,
                        config,
                        update,
                        phase_label,
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )

    # END OF TRAINING
    probe_and_save_codebook(env, params, rng, doer, config["fsq_levels"])

def probe_and_save_codebook(env, params, rng, doer, fsq_levels):
    import itertools
    import json
    
    print("\n" + "="*72)
    print("PROBING CODEBOOK AT END OF TRAINING")
    print("="*72)
    
    ranges = [list(range(l)) for l in fsq_levels]
    words = list(itertools.product(*ranges))
    
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)
    
    dummy_local = jnp.zeros((1, 3, 3, 5), dtype=jnp.float32)
    dummy_prop = jnp.zeros((1, env.num_doers + 3), dtype=jnp.float32)
    rng, key = jax.random.split(rng)
    bank = env._build_item_bank()
    dummy_menu = jnp.stack([bank[0], bank[1], bank[2], bank[3]])[None, ...]
    
    codebook_mapping = {}
    
    for word in words:
        message_array = jnp.array([word], dtype=jnp.float32)
        _, action_logits = doer.apply(
            {"params": params["doer"]},
            doer_carry,
            dummy_local,
            dummy_prop,
            message_array,
            dummy_menu
        )
        action = int(jnp.argmax(action_logits[0]))
        if action < 5:
            action_desc = ["stay", "up", "right", "down", "left"][action]
        else:
            action_desc = f"pick_option_{action-5}"
        
        word_str = str(word)
        codebook_mapping[word_str] = action_desc
        print(f"Message {word_str} -> Doer Action: {action_desc}")
        
    with open("final_codebook.json", "w") as f:
        json.dump(codebook_mapping, f, indent=4)
        
    try:
        from flax.training import checkpoints
        import os
        os.makedirs("checkpoints", exist_ok=True)
        checkpoints.save_checkpoint(ckpt_dir="checkpoints/", target=params, step=1000, overwrite=True)
        print("Saved final model weights to checkpoints/")
    except ImportError:
        pass
    print("Codebook saved to final_codebook.json")

if __name__ == "__main__":
    main()





In [ ]:
!python train.py